# RDBScore V3 — Bedrock Execution and Results

This notebook contains only the experimental execution side of RDBScore V2: loading the benchmark instances generated by the exploration notebook, invoking Amazon Bedrock, consolidating model outputs, LLM-as-Judge adjudication, and final result aggregation.

It does **not** rediscover candidates or regenerate mutations.


## 1. Environment and Bedrock configuration


In [1]:
import os
import sys
import json
import time
from pathlib import Path
from collections import Counter
from datetime import datetime, timezone

import yaml
import boto3
import botocore

print("Python:   ", sys.version)
print("boto3:    ", boto3.__version__)
print("botocore: ", botocore.__version__)


Python:    3.7.9 (default, Jun 24 2022, 18:14:15) 
[GCC 8.5.0 20210514 (Red Hat 8.5.0-13)]
boto3:     1.33.13
botocore:  1.33.13


### AWS credentials and clients

The following cell is preserved **exactly as provided in the source notebook**, as requested.


In [114]:
# AWS temporary credentials - fill locally before creating the Bedrock client
AWS_ACCESS_KEY_ID = "..."
AWS_SECRET_ACCESS_KEY = "..."
AWS_SESSION_TOKEN = "..."

bedrock = boto3.client(
    "bedrock-runtime",
    region_name="sa-east-1",
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    aws_session_token=AWS_SESSION_TOKEN
)

bedrock_control = boto3.client(
    "bedrock",
    region_name="sa-east-1",
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    aws_session_token=AWS_SESSION_TOKEN
)

print("Runtime:", bedrock.meta.service_model.service_name)
print("Control:", bedrock_control.meta.service_model.service_name)
print("Região:", bedrock.meta.region_name)

Runtime: bedrock-runtime
Control: bedrock
Região: sa-east-1


### 2. Load the frozen benchmark artifacts

Run the exploration/construction notebook first. This execution notebook consumes its manifests and generated audit inputs.


In [115]:
MODEL_ID = "anthropic.claude-3-sonnet-20240229-v1:0"

body = {
    "anthropic_version": "bedrock-2023-05-31",
    "max_tokens": 10,
    "temperature": 0,
    "messages": [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "Reply with exactly: OK"
                }
            ]
        }
    ]
}

response = bedrock.invoke_model(
    modelId=MODEL_ID,
    body=json.dumps(body),
    contentType="application/json",
    accept="application/json"
)

result = json.loads(response["body"].read())

print(result)

{'id': 'msg_bdrk_01F7n3uKQW88zwemHczWSdDH', 'type': 'message', 'role': 'assistant', 'model': 'claude-3-sonnet-20240229', 'content': [{'type': 'text', 'text': 'OK'}], 'stop_reason': 'end_turn', 'stop_sequence': None, 'usage': {'input_tokens': 12, 'output_tokens': 4}}


In [116]:
# Enter the Spider2 working directory when starting from the parent directory.
if Path("Spider2").exists() and not Path("spider2-dbt").exists():
    os.chdir("Spider2")

RDBSCORE_OUTPUT_V2 = Path("rdbscore_generated_v2")
manifest_v2_path = RDBSCORE_OUTPUT_V2 / "generation_manifest.json"
manifest_l3_path = RDBSCORE_OUTPUT_V2 / "L3" / "generation_manifest_l3.json"

for required in [manifest_v2_path, manifest_l3_path]:
    if not required.exists():
        raise FileNotFoundError(
            "Missing benchmark artifact: {}. Run the exploration/construction notebook first.".format(required)
        )

generation_manifest_v2 = json.loads(manifest_v2_path.read_text(encoding="utf-8"))
generation_manifest_l3 = json.loads(manifest_l3_path.read_text(encoding="utf-8"))

print("L1/L2 manifest records:", len(generation_manifest_v2))
print("L3 manifest records:   ", len(generation_manifest_l3))


L1/L2 manifest records: 1510
L3 manifest records:    51


## 3. V3 — Quality Expectation Reconstruction

The first V3 experiment investigates whether a neural model can
reconstruct an explicit schema quality expectation from the evidence
available to the auditor, without being asked to identify a defect.

The experiment initially considers one deterministic instance of
L1_REMOVE_UNIQUE. Ground-truth mutation metadata is excluded from
the model-visible input.

In [117]:
# Inspect one ground-truth record before selecting V3 cases

gt_files = sorted(RDBSCORE_OUTPUT_V2.rglob("ground_truth.json"))

print("Ground-truth files:", len(gt_files))

if not gt_files:
    raise RuntimeError(
        f"No ground_truth.json files found under {RDBSCORE_OUTPUT_V2}"
    )

sample_gt_path = gt_files[0]
sample_gt = json.loads(sample_gt_path.read_text(encoding="utf-8"))

print("\nSample path:")
print(sample_gt_path)

print("\nTop-level keys:")
print(list(sample_gt.keys()))

print("\nSample ground truth:")
print(json.dumps(sample_gt, indent=2, ensure_ascii=False))

Ground-truth files: 1561

Sample path:
rdbscore_generated_v2/L1/L1_REMOVE_NOT_NULL/L1_REMOVE_NOT_NULL_0057fea9a08f/ground_truth.json

Top-level keys:
['mutation_id', 'operator_id', 'level', 'defect_family', 'mutation_type', 'expected_defect', 'evidence_requirement', 'benchmark', 'protocol_version', 'project', 'artifact', 'candidate', 'evidence']

Sample ground truth:
{
  "mutation_id": "L1_REMOVE_NOT_NULL_0057fea9a08f",
  "operator_id": "L1_REMOVE_NOT_NULL",
  "level": "L1",
  "defect_family": "Constraint Defects",
  "mutation_type": "REMOVE_NOT_NULL",
  "expected_defect": "Missing explicitly declared non-null constraint/test.",
  "evidence_requirement": "EXPLICIT_METADATA",
  "benchmark": "Spider2-DBT",
  "protocol_version": "RDBSCORE_V2",
  "project": "airport001",
  "artifact": "models/core/core.yml",
  "candidate": {
    "model": "fct_arrivals__malaysia_summary",
    "column": "airport_id"
  },
  "evidence": null
}


In [119]:
# Select one deterministic L1 case for the first V3 experiment

V3_OPERATOR = "L1_REMOVE_UNIQUE"

gt_files = sorted(RDBSCORE_OUTPUT_V2.rglob("ground_truth.json"))

v3_candidates = []

for gt_path in gt_files:
    gt = json.loads(gt_path.read_text(encoding="utf-8"))

    if gt.get("operator_id") == V3_OPERATOR:
        v3_candidates.append(gt_path)

v3_candidates = sorted(v3_candidates, key=lambda p: str(p))

print("Operator:", V3_OPERATOR)
print("Candidates:", len(v3_candidates))

if not v3_candidates:
    raise RuntimeError(f"No candidates found for {V3_OPERATOR}")

# Deterministic selection: first candidate in lexical path order
v3_case_gt_path = v3_candidates[0]
v3_case_dir = v3_case_gt_path.parent
v3_audit_input_path = v3_case_dir / "audit_input.txt"

print("\nSelected case:")
print(v3_case_dir)

print("\nAudit input exists:")
print(v3_audit_input_path.exists())

Operator: L1_REMOVE_UNIQUE
Candidates: 300

Selected case:
rdbscore_generated_v2/L1/L1_REMOVE_UNIQUE/L1_REMOVE_UNIQUE_0088503f05f3

Audit input exists:
True


In [120]:
# Inspect the model-visible evidence for the selected V3 case

v3_audit_input = v3_audit_input_path.read_text(encoding="utf-8")

print("=== MODEL-VISIBLE EVIDENCE ===\n")
print(v3_audit_input)

=== MODEL-VISIBLE EVIDENCE ===

SCHEMA METADATA

version: 2
models:
- name: int_lever__contact_info
  description: Table that the most relevant contact info associated with each contact.
  columns:
  - name: contact_id
    description: Unique ID of the contact.
    tests:
    - unique
    - not_null
  - name: emails
    description: Aggregated list of email addresses associated with this contact.
  - name: phones
    description: Aggregated list of the contact's phone numbers (with phone types).
  - name: linkedin_link
    description: Contact URL that contains 'linkedin.com' in it.
  - name: github_link
    description: Contact URL that contains 'github.com' in it.
- name: int_lever__interview_feedback
  description: Table associating interviews with feedback given by individual interviewers.
    A single interview might have submit multiple feedback forms.
  tests:
  - dbt_utils.unique_combination_of_columns:
      combination_of_columns:
      - feedback_form_id
      - interview_id

### 3.1 Quality Expectation Reconstruction — L1

The neural component does not audit the schema and does not determine
whether a defect exists.

Its task is restricted to reconstructing schema quality expectations
that are supported by the available evidence.

For the initial L1 experiment, the target expectation type is:

    EXPECTED_UNIQUE(model, column)

The subsequent verification of whether the expected uniqueness
constraint is actually declared in the schema is deterministic and
is not performed by the neural component.

In [121]:
# V3 prompt — reconstruct EXPECTED_UNIQUE expectations

V3_EXPECTATION_PROMPT = """You are analyzing a relational database project.

Your task is NOT to audit the schema for defects.

Your task is to reconstruct quality expectations about attribute
uniqueness from the available schema documentation and SQL definitions.

Identify attributes or combinations of attributes for which the
available evidence supports the expectation that they should be
unique in their model.

Do not determine whether the expectation is currently satisfied.
Do not report missing constraints or schema defects.
Do not assume that columns ending in "_id" must be unique.
A foreign-key relationship does NOT by itself imply that the
referencing attribute should be unique.

For each supported expectation, return:

MODEL: <model name>
COLUMNS: <one or more column names>
EXPECTATION: EXPECTED_UNIQUE
EVIDENCE: <specific evidence supporting the expectation>
CONFIDENCE: HIGH | MEDIUM | LOW

Only return an expectation when it is supported by evidence.
If no uniqueness expectation can be established, return:

NO_EXPECTATION
"""

In [122]:
# Inspect available Bedrock helper functions

import inspect

# Minimal Claude Sonnet invocation for V3

import json

SONNET_MODEL_ID = "anthropic.claude-3-sonnet-20240229-v1:0"

def invoke_v3_sonnet(system_prompt, evidence, max_tokens=2000):
    body = {
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": max_tokens,
        "temperature": 0,
        "system": system_prompt,
        "messages": [
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": evidence
                    }
                ]
            }
        ]
    }

    response = bedrock_runtime.invoke_model(
        modelId=SONNET_MODEL_ID,
        body=json.dumps(body)
    )

    result = json.loads(response["body"].read())

    return result["content"][0]["text"]

for name, obj in sorted(globals().items()):
    if callable(obj) and any(
        term in name.lower()
        for term in ["bedrock", "claude", "sonnet", "invoke"]
    ):
        print(name)

invoke_v3_sonnet


In [123]:
# Bedrock Runtime client for V3

bedrock_runtime = boto3.client(
    "bedrock-runtime",
    region_name="sa-east-1",
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    aws_session_token=AWS_SESSION_TOKEN,
)

print("Bedrock Runtime client ready.")

Bedrock Runtime client ready.


In [14]:
# First V3 neural expectation reconstruction

v3_response = invoke_v3_sonnet(
    system_prompt=V3_EXPECTATION_PROMPT,
    evidence=v3_audit_input,
    max_tokens=2000,
)

print("=== V3 QUALITY EXPECTATIONS ===\n")
print(v3_response)

=== V3 QUALITY EXPECTATIONS ===

Based on the schema metadata and SQL model definitions, I can identify the following expectations for unique attributes or combinations of attributes:

MODEL: int_lever__contact_info
COLUMNS: contact_id
EXPECTATION: EXPECTED_UNIQUE
EVIDENCE: The column contact_id is described as a "Unique ID of the contact" and has a unique test defined.
CONFIDENCE: HIGH

MODEL: int_lever__interview_feedback
COLUMNS: feedback_form_id, interview_id
EXPECTATION: EXPECTED_UNIQUE
EVIDENCE: The model has a dbt_utils.unique_combination_of_columns test defined for the combination of feedback_form_id and interview_id.
CONFIDENCE: HIGH

MODEL: int_lever__opportunity_application
COLUMNS: opportunity_id
EXPECTATION: EXPECTED_UNIQUE
EVIDENCE: The column opportunity_id is described as a "Unique ID of the opportunity" and has unique and not_null tests defined.
CONFIDENCE: HIGH

MODEL: int_lever__opportunity_contact_info
COLUMNS: opportunity_id
EXPECTATION: EXPECTED_UNIQUE
EVIDENCE: T

### 3.2 Deterministic Extraction of Explicit Expectations

Explicit uniqueness expectations are extracted directly from schema
metadata without neural interpretation.

The extractor recognizes both single-column `unique` tests and
`dbt_utils.unique_combination_of_columns` tests, producing the same
logical representation used for reconstructed expectations:

    EXPECTED_UNIQUE(model, {column_1, ..., column_n})

This separates expectations already represented in the schema from
expectations that must be reconstructed from other available evidence.

In [15]:
# Deterministically extract explicit uniqueness expectations

import yaml

def extract_schema_metadata(audit_input):
    marker_start = "SCHEMA METADATA"
    marker_end = "========================================"

    start = audit_input.find(marker_start)
    if start == -1:
        raise ValueError("SCHEMA METADATA section not found")

    start = audit_input.find("\n", start) + 1

    end = audit_input.find(marker_end, start)

    # Some benchmark inputs end directly after the YAML metadata.
    if end == -1:
        end = len(audit_input)

    metadata_text = audit_input[start:end].strip()

    # Remove optional separator immediately after the heading.
    if metadata_text.startswith("==============="):
        metadata_text = metadata_text.split("\n", 1)[1].strip()

    return yaml.safe_load(metadata_text)


def extract_explicit_unique_expectations(audit_input):
    metadata = extract_schema_metadata(audit_input)
    expectations = []

    for model in metadata.get("models", []):
        model_name = model.get("name")

        # Single-column uniqueness
        for column in model.get("columns", []):
            column_name = column.get("name")

            for test in column.get("tests", []) or []:
                if test == "unique":
                    expectations.append({
                        "model": model_name,
                        "columns": (column_name,),
                        "expectation": "EXPECTED_UNIQUE",
                        "source": "EXPLICIT_METADATA"
                    })

        # Composite uniqueness
        for test in model.get("tests", []) or []:
            if isinstance(test, dict):
                unique_test = test.get(
                    "dbt_utils.unique_combination_of_columns"
                )

                if unique_test:
                    columns = tuple(
                        unique_test.get("combination_of_columns", [])
                    )

                    expectations.append({
                        "model": model_name,
                        "columns": columns,
                        "expectation": "EXPECTED_UNIQUE",
                        "source": "EXPLICIT_METADATA"
                    })

    return expectations


q_explicit = extract_explicit_unique_expectations(v3_audit_input)

print("Explicit uniqueness expectations:", len(q_explicit))
print()

for q in q_explicit:
    print(
        f"{q['model']} | "
        f"{', '.join(q['columns'])}"
    )

Explicit uniqueness expectations: 9

int_lever__contact_info | contact_id
int_lever__interview_feedback | feedback_form_id, interview_id
int_lever__opportunity_application | opportunity_id
int_lever__opportunity_contact_info | opportunity_id
int_lever__opportunity_tags | opportunity_id
int_lever__opportunity_users | opportunity_id
int_lever__posting_applications | posting_id
int_lever__posting_interviews | posting_id
int_lever__posting_tags | posting_id


In [16]:
# V3 prompt — reconstruct EXPECTED_UNIQUE expectations

V3_EXPECTATION_PROMPT = """You are analyzing a relational database project.

Your task is NOT to audit the schema for defects.

Your task is to reconstruct quality expectations about attribute
uniqueness from the available schema documentation and SQL definitions.

Identify attributes or combinations of attributes for which the
available evidence supports the expectation that they should be
unique in their model.

Do not determine whether the expectation is currently satisfied.
Do not report missing constraints or schema defects.
Do not assume that columns ending in "_id" must be unique.
A foreign-key relationship does NOT by itself imply that the
referencing attribute should be unique.

Return only valid JSON using this structure:

{
  "expectations": [
    {
      "model": "<model name>",
      "columns": ["<column name>", "..."],
      "expectation": "EXPECTED_UNIQUE",
      "evidence": "<specific evidence supporting the expectation>",
      "confidence": "HIGH | MEDIUM | LOW"
    }
  ]
}

If no uniqueness expectation can be established, return:

{
  "expectations": []
}

Do not include markdown, explanations, or any text outside the JSON object.
"""

In [17]:
v3_response = invoke_v3_sonnet(
    system_prompt=V3_EXPECTATION_PROMPT,
    evidence=v3_audit_input,
    max_tokens=2000,
)

print("=== V3 QUALITY EXPECTATIONS ===\n")
print(v3_response)

=== V3 QUALITY EXPECTATIONS ===

{
  "expectations": [
    {
      "model": "int_lever__contact_info",
      "columns": ["contact_id"],
      "expectation": "EXPECTED_UNIQUE",
      "evidence": "Column 'contact_id' has a 'unique' test defined in the schema metadata.",
      "confidence": "HIGH"
    },
    {
      "model": "int_lever__interview_feedback",
      "columns": ["feedback_form_id", "interview_id"],
      "expectation": "EXPECTED_UNIQUE",
      "evidence": "The schema metadata includes a dbt_utils.unique_combination_of_columns test for the combination of 'feedback_form_id' and 'interview_id'.",
      "confidence": "HIGH"
    },
    {
      "model": "int_lever__opportunity_application",
      "columns": ["opportunity_id"],
      "expectation": "EXPECTED_UNIQUE",
      "evidence": "Column 'opportunity_id' has 'unique' and 'not_null' tests defined in the schema metadata.",
      "confidence": "HIGH"
    },
    {
      "model": "int_lever__opportunity_contact_info",
      "columns

### 3.3 Deterministic Expectation Verification

Neurally reconstructed expectations are compared with expectations
explicitly represented in the schema metadata.

For uniqueness expectations, the comparison is defined as:

    Q_missing = Q_expected - Q_explicit

An expectation in `Q_missing` represents a quality property supported
by the available evidence but not explicitly enforced by the schema
metadata. The comparison itself requires no neural interpretation.

In [18]:
# Compare reconstructed expectations with explicit metadata

v3_result = json.loads(v3_response)

q_expected = v3_result.get("expectations", [])


def expectation_key(q):
    return (
        q["expectation"],
        q["model"],
        tuple(sorted(q["columns"]))
    )


explicit_keys = {
    expectation_key(q)
    for q in q_explicit
}

q_missing = [
    q for q in q_expected
    if expectation_key(q) not in explicit_keys
]


print("Reconstructed expectations:", len(q_expected))
print("Explicit expectations:", len(q_explicit))
print("Missing expectations:", len(q_missing))

print("\n=== MISSING EXPECTATIONS ===\n")

for q in q_missing:
    print("MODEL:", q["model"])
    print("COLUMNS:", ", ".join(q["columns"]))
    print("EXPECTATION:", q["expectation"])
    print("EVIDENCE:", q["evidence"])
    print("CONFIDENCE:", q["confidence"])
    print()

Reconstructed expectations: 10
Explicit expectations: 9
Missing expectations: 1

=== MISSING EXPECTATIONS ===

MODEL: int_lever__requisition_users
COLUMNS: requisition_id
EXPECTATION: EXPECTED_UNIQUE
EVIDENCE: Column 'requisition_id' has a 'not_null' test defined in the schema metadata.
CONFIDENCE: MEDIUM



In [19]:
# Reveal ground truth only after V3 inference is complete

v3_ground_truth = json.loads(
    v3_case_gt_path.read_text(encoding="utf-8")
)

print("=== GROUND TRUTH ===\n")
print(json.dumps(v3_ground_truth, indent=2, ensure_ascii=False))

=== GROUND TRUTH ===

{
  "mutation_id": "L1_REMOVE_UNIQUE_0088503f05f3",
  "operator_id": "L1_REMOVE_UNIQUE",
  "level": "L1",
  "defect_family": "Constraint Defects",
  "mutation_type": "REMOVE_UNIQUE",
  "expected_defect": "Missing explicitly declared uniqueness constraint/test.",
  "evidence_requirement": "EXPLICIT_METADATA",
  "benchmark": "Spider2-DBT",
  "protocol_version": "RDBSCORE_V2",
  "project": "lever001",
  "artifact": "models/intermediate/int_lever.yml",
  "candidate": {
    "model": "int_lever__requisition_users",
    "column": "requisition_id"
  },
  "evidence": null
}


## 4. V3 — L2 Quality Expectation Reconstruction

The second V3 experiment investigates the form of quality expectations
required to establish normalization design flaws.

We initially consider one deterministic instance of
`L2_INTRODUCE_TRANSITIVE_DEPENDENCY`.

As in the L1 experiment, ground-truth mutation metadata is used only
for deterministic case selection and subsequent evaluation. It is
excluded from the evidence available to the neural component.

In [20]:
# Select one deterministic L2 case

V3_L2_OPERATOR = "L2_INTRODUCE_TRANSITIVE_DEPENDENCY"

v3_l2_candidates = []

for gt_path in gt_files:
    gt = json.loads(gt_path.read_text(encoding="utf-8"))

    if gt.get("operator_id") == V3_L2_OPERATOR:
        v3_l2_candidates.append(gt_path)

v3_l2_candidates = sorted(
    v3_l2_candidates,
    key=lambda p: str(p)
)

print("Operator:", V3_L2_OPERATOR)
print("Candidates:", len(v3_l2_candidates))

if not v3_l2_candidates:
    raise RuntimeError(
        f"No candidates found for {V3_L2_OPERATOR}"
    )

v3_l2_gt_path = v3_l2_candidates[0]
v3_l2_case_dir = v3_l2_gt_path.parent
v3_l2_audit_input_path = v3_l2_case_dir / "audit_input.txt"

print("\nSelected case:")
print(v3_l2_case_dir)

print("\nAudit input exists:")
print(v3_l2_audit_input_path.exists())

Operator: L2_INTRODUCE_TRANSITIVE_DEPENDENCY
Candidates: 349

Selected case:
rdbscore_generated_v2/L2/L2_INTRODUCE_TRANSITIVE_DEPENDENCY/L2_INTRODUCE_TRANSITIVE_DEPENDENCY_013b469473fc

Audit input exists:
True


In [21]:
# Inspect the model-visible evidence for the selected L2 case

v3_l2_audit_input = v3_l2_audit_input_path.read_text(
    encoding="utf-8"
)

print("=== L2 MODEL-VISIBLE EVIDENCE ===\n")
print(v3_l2_audit_input)

=== L2 MODEL-VISIBLE EVIDENCE ===

SCHEMA METADATA

models:
- name: stg_synthea__patients
  description: Synthea patients table
  columns:
  - name: patient_id
    data_type: text
    tests:
    - unique
  - name: birth_date
    data_type: date
  - name: death_date
    data_type: date
  - name: ssn
    data_type: text
  - name: drivers_license_number
    data_type: text
  - name: passport_number
    data_type: text
  - name: patient_prefix
    data_type: text
    description: Name prefix, such as Mr., Mrs., Dr., etc.
  - name: patient_first_name
    data_type: text
  - name: patient_last_name
    data_type: text
  - name: patient_suffix
    data_type: text
    description: Name suffix, such as PhD, MD, JD, etc.
  - name: patient_maiden_name
    data_type: text
  - name: marital_status
    description: Marital Status. M is married, S is single. Currently no support for
      divorce (D) or widowing (W).
    data_type: text
  - name: race
    description: Text description of patient's pr

In [22]:
# Reveal L2 ground truth after manual evidence analysis

v3_l2_ground_truth = json.loads(
    v3_l2_gt_path.read_text(encoding="utf-8")
)

print("=== L2 GROUND TRUTH ===\n")
print(
    json.dumps(
        v3_l2_ground_truth,
        indent=2,
        ensure_ascii=False
    )
)

=== L2 GROUND TRUTH ===

{
  "mutation_id": "L2_INTRODUCE_TRANSITIVE_DEPENDENCY_013b469473fc",
  "operator_id": "L2_INTRODUCE_TRANSITIVE_DEPENDENCY",
  "level": "L2",
  "defect_family": "Normalization Design Flaws",
  "mutation_type": "INTRODUCE_TRANSITIVE_DEPENDENCY",
  "expected_defect": "Transitive dependency introduced through an explicit dependency chain.",
  "evidence_requirement": "NORMALIZATION_DEPENDENCY_INFERENCE",
  "benchmark": "Spider2-DBT",
  "protocol_version": "RDBSCORE_V2",
  "project": "synthea001",
  "artifact": "models/staging/synthea/_synthea__models.yml",
  "candidate": {
    "source_model": "stg_synthea__providers",
    "source_key": "provider_id",
    "source_fk": "organization_id",
    "target_model": "stg_synthea__organizations",
    "target_key": "organization_id",
    "target_column": "organization_state",
    "evidence": {
      "type": "TRANSITIVE_DEPENDENCY_EVIDENCE",
      "source_key_unique": true,
      "source_fk_unique": false,
      "relationship": 

## 5. V3 — L3 Semantic Knowledge Reconstruction

The third V3 experiment investigates the form of design knowledge
required to establish semantic schema defects.

We initially consider one deterministic instance of
L3_INTRODUCE_SEMANTIC_RELATIONSHIP_MISMATCH.

As in the previous experiments, ground-truth mutation metadata is used
only for deterministic case selection and subsequent evaluation. It is
excluded from the evidence available during semantic reconstruction.

The immediate objective is not to detect the defect, but to determine
which semantic knowledge must be made explicit before the relationship
can be verified.

In [23]:
# Select one deterministic L3 case

V3_L3_OPERATOR = "L3_INTRODUCE_SEMANTIC_RELATIONSHIP_MISMATCH"

v3_l3_candidates = []

for gt_path in gt_files:
    gt = json.loads(gt_path.read_text(encoding="utf-8"))

    if gt.get("operator_id") == V3_L3_OPERATOR:
        v3_l3_candidates.append(gt_path)

v3_l3_candidates = sorted(
    v3_l3_candidates,
    key=lambda p: str(p)
)

print("Operator:", V3_L3_OPERATOR)
print("Candidates:", len(v3_l3_candidates))

if not v3_l3_candidates:
    raise RuntimeError(
        f"No candidates found for {V3_L3_OPERATOR}"
    )

v3_l3_gt_path = v3_l3_candidates[0]
v3_l3_case_dir = v3_l3_gt_path.parent
v3_l3_audit_input_path = v3_l3_case_dir / "audit_input.txt"

print("\nSelected case:")
print(v3_l3_case_dir)

print("\nAudit input exists:")
print(v3_l3_audit_input_path.exists())

Operator: L3_INTRODUCE_SEMANTIC_RELATIONSHIP_MISMATCH
Candidates: 51

Selected case:
rdbscore_generated_v2/L3/L3_REL_0001_airbnb001_dim_listings_hosts_host_id

Audit input exists:
True


In [24]:
# Inspect the model-visible evidence for the selected L3 case

v3_l3_audit_input = v3_l3_audit_input_path.read_text(
    encoding="utf-8"
)

print("=== L3 MODEL-VISIBLE EVIDENCE ===\n")
print(v3_l3_audit_input)

=== L3 MODEL-VISIBLE EVIDENCE ===

You are auditing a relational database schema.

Identify semantic schema problems that are directly justified by the
schema metadata provided below.

Focus on whether columns, relationships, and referenced entities are
semantically coherent with one another.

A relationship may be structurally valid while connecting concepts
that do not correspond semantically.

Use only the evidence present in the supplied schema metadata.
Do not invent domain requirements that are not supported by the schema.

Return ONLY valid JSON with this structure:

{
  "findings": [
    {
      "defect_type": "...",
      "affected_element": "...",
      "evidence": "...",
      "reasoning": "..."
    }
  ]
}

If no justified problem can be identified, return:

{"findings":[]}


SCHEMA METADATA:

version: 2
models:
- name: daily_agg_reviews
  columns:
  - name: DATE_SENTIMENT_ID
    tests:
    - unique
    - not_null
  - name: REVIEW_SENTIMENT
    tests:
    - accepted_values:

In [25]:
# Reveal L3 ground truth after manual semantic analysis

v3_l3_ground_truth = json.loads(
    v3_l3_gt_path.read_text(encoding="utf-8")
)

print("=== L3 GROUND TRUTH ===\n")
print(
    json.dumps(
        v3_l3_ground_truth,
        indent=2,
        ensure_ascii=False
    )
)

=== L3 GROUND TRUTH ===

{
  "protocol_version": "RDBSCORE_V2",
  "level": "L3",
  "operator_id": "L3_INTRODUCE_SEMANTIC_RELATIONSHIP_MISMATCH",
  "defect_family": "Semantic Schema Defects",
  "defect_type": "Semantic Relationship Mismatch",
  "project": "airbnb001",
  "source_model": "dim_listings_hosts",
  "source_column": "host_id",
  "semantic_expectation": {
    "intended_target_entity": "dim_hosts",
    "intended_target_key": "host_id"
  },
  "evidence": {
    "original_relationship": "dim_listings_hosts.host_id -> dim_hosts.host_id",
    "evidence_type": "explicit original semantic correspondence"
  },
  "mutation": {
    "original_target": "dim_hosts",
    "original_target_key": "host_id",
    "mutated_target": "wow_agg_reviews",
    "mutated_target_key": "date_sentiment_id",
    "mutated_target_key_not_null": true,
    "target_semantic_similarity": 0.0,
    "alternative_count": 5,
    "selection_index": 4,
    "selection_method": "SHA256 deterministic diversified selection"
  

## 6. Minimal Schema Design Knowledge

The exploratory L1–L3 cases suggest that schema auditing does not
require all relevant design knowledge to be reconstructed by a neural
component.

We distinguish three forms of knowledge:

    K_explicit       — facts directly extractable from schema metadata
    K_reconstructed  — design expectations requiring semantic interpretation
    K_derived        — facts inferred deterministically from other knowledge

The first experiment investigates whether the functional dependencies
required for the L2 transitive-dependency case can be derived from
explicit schema metadata, without neural interpretation.

### 6.1 Deriving Functional Dependencies

For a model M with a unique key K, every attribute A represented in M
is functionally dependent on K:

    Unique(M, K) ∧ Attribute(M, A)
        → FD(M.K, M.A)

For a relationship from M.A to N.K, where N.K is unique, attributes B
of N can be functionally determined through the referenced key:

    References(M.A, N.K)
    ∧ Unique(N, K)
    ∧ Attribute(N, B)
        → FD(M.A, N.B)

A transitive dependency candidate can therefore be derived when:

    FD(M.K, M.A)
    ∧ FD(M.A, N.B)
    ∧ Attribute(M, B)
        → TransitiveDependency(M.K, M.A, M.B)

The rules operate only over explicit schema facts and do not require
semantic interpretation by the neural component.

In [26]:
# Extract explicit schema facts for the selected L2 case

l2_metadata = extract_schema_metadata(v3_l2_audit_input)

l2_models = {
    model["name"]: model
    for model in l2_metadata.get("models", [])
}

source_model = l2_models["stg_synthea__providers"]
target_model = l2_models["stg_synthea__organizations"]

source_columns = {
    column["name"]
    for column in source_model.get("columns", [])
}

target_columns = {
    column["name"]
    for column in target_model.get("columns", [])
}

shared_columns = sorted(
    source_columns.intersection(target_columns)
)

print("Source model:", source_model["name"])
print("Target model:", target_model["name"])

print("\nShared column names:")
for column in shared_columns:
    print("-", column)

Source model: stg_synthea__providers
Target model: stg_synthea__organizations

Shared column names:
- organization_id
- organization_state


In [27]:
# Generate candidate semantic correspondences for the L2 case

relationship_column = "organization_id"

candidate_correspondences = [
    {
        "source_model": source_model["name"],
        "source_column": column,
        "target_model": target_model["name"],
        "target_column": column
    }
    for column in shared_columns
    if column != relationship_column
]

print("=== CANDIDATE SEMANTIC CORRESPONDENCES ===\n")

for candidate in candidate_correspondences:
    print(
        f'{candidate["source_model"]}.{candidate["source_column"]}'
        " <-> "
        f'{candidate["target_model"]}.{candidate["target_column"]}'
    )

=== CANDIDATE SEMANTIC CORRESPONDENCES ===

stg_synthea__providers.organization_state <-> stg_synthea__organizations.organization_state


In [28]:
# Build addressable evidence units for the L2 correspondence

l2_evidence_units = [
    {
        "id": "E001",
        "type": "ATTRIBUTE",
        "model": "stg_synthea__providers",
        "column": "organization_state"
    },
    {
        "id": "E002",
        "type": "ATTRIBUTE",
        "model": "stg_synthea__organizations",
        "column": "organization_state"
    },
    {
        "id": "E003",
        "type": "RELATIONSHIP",
        "source_model": "stg_synthea__providers",
        "source_column": "organization_id",
        "target_model": "stg_synthea__organizations",
        "target_column": "organization_id"
    },
    {
        "id": "E004",
        "type": "UNIQUE",
        "model": "stg_synthea__organizations",
        "column": "organization_id"
    }
]

print("=== L2 ADDRESSABLE EVIDENCE ===\n")

for evidence in l2_evidence_units:
    print(
        evidence["id"],
        json.dumps(evidence, ensure_ascii=False)
    )

=== L2 ADDRESSABLE EVIDENCE ===

E001 {"id": "E001", "type": "ATTRIBUTE", "model": "stg_synthea__providers", "column": "organization_state"}
E002 {"id": "E002", "type": "ATTRIBUTE", "model": "stg_synthea__organizations", "column": "organization_state"}
E003 {"id": "E003", "type": "RELATIONSHIP", "source_model": "stg_synthea__providers", "source_column": "organization_id", "target_model": "stg_synthea__organizations", "target_column": "organization_id"}
E004 {"id": "E004", "type": "UNIQUE", "model": "stg_synthea__organizations", "column": "organization_id"}


### 6.2 Semantic Correspondence Reconstruction

Candidate attribute correspondences can be generated deterministically
from structural evidence, but structural similarity alone does not
establish semantic equivalence.

The neural component is therefore restricted to determining whether a
candidate pair represents the same conceptual property.

For each candidate pair, the output is:

    SEMANTIC_CORRESPONDENCE(source, target)

only when the available evidence supports that interpretation.

The neural component does not evaluate normalization, functional
dependencies, transitive dependencies, or schema defects.

In [29]:
# V3 prompt — validate candidate semantic correspondences using addressable evidence

V3_CORRESPONDENCE_PROMPT = """You are analyzing relational schema evidence.

Your task is NOT to audit the schema for defects.

You will receive one candidate correspondence between two attributes
and a set of identified evidence units.

Determine whether the two attributes represent the same conceptual
property in the domain.

Use only the supplied evidence units.

If you conclude that a semantic correspondence exists, identify the
evidence units supporting the conclusion using their evidence IDs.

Do not invent descriptions, documentation, relationships, constraints,
or domain information that is not present in the supplied evidence.

Do not evaluate normalization.
Do not identify functional dependencies.
Do not identify transitive dependencies.
Do not report schema defects.

Return only valid JSON using this structure:

{
  "semantic_correspondence": true,
  "evidence_ids": ["E001", "E002"],
  "confidence": "HIGH | MEDIUM | LOW"
}

If the available evidence does not support the correspondence, return:

{
  "semantic_correspondence": false,
  "evidence_ids": [],
  "confidence": "HIGH | MEDIUM | LOW"
}

Do not include markdown or text outside the JSON object.
"""


candidate = candidate_correspondences[0]

candidate_payload = {
    "candidate_correspondence": {
        "source": {
            "model": candidate["source_model"],
            "column": candidate["source_column"]
        },
        "target": {
            "model": candidate["target_model"],
            "column": candidate["target_column"]
        }
    },
    "evidence_units": l2_evidence_units
}

candidate_text = json.dumps(
    candidate_payload,
    indent=2
)

print("=== MODEL-VISIBLE INPUT ===\n")
print(candidate_text)

v3_correspondence_response = invoke_v3_sonnet(
    V3_CORRESPONDENCE_PROMPT,
    candidate_text
)

print("\n=== SONNET RESPONSE ===\n")
print(v3_correspondence_response)

=== MODEL-VISIBLE INPUT ===

{
  "candidate_correspondence": {
    "source": {
      "model": "stg_synthea__providers",
      "column": "organization_state"
    },
    "target": {
      "model": "stg_synthea__organizations",
      "column": "organization_state"
    }
  },
  "evidence_units": [
    {
      "id": "E001",
      "type": "ATTRIBUTE",
      "model": "stg_synthea__providers",
      "column": "organization_state"
    },
    {
      "id": "E002",
      "type": "ATTRIBUTE",
      "model": "stg_synthea__organizations",
      "column": "organization_state"
    },
    {
      "id": "E003",
      "type": "RELATIONSHIP",
      "source_model": "stg_synthea__providers",
      "source_column": "organization_id",
      "target_model": "stg_synthea__organizations",
      "target_column": "organization_id"
    },
    {
      "id": "E004",
      "type": "UNIQUE",
      "model": "stg_synthea__organizations",
      "column": "organization_id"
    }
  ]
}

=== SONNET RESPONSE ===

{
  "semanti

In [30]:
# Deterministically validate neural evidence references

correspondence_result = json.loads(v3_correspondence_response)

available_evidence_ids = {
    evidence["id"]
    for evidence in l2_evidence_units
}

claimed_evidence_ids = set(
    correspondence_result.get("evidence_ids", [])
)

unknown_evidence_ids = (
    claimed_evidence_ids - available_evidence_ids
)

provenance_valid = (
    len(claimed_evidence_ids) > 0
    and len(unknown_evidence_ids) == 0
)

print("Claimed evidence IDs:", sorted(claimed_evidence_ids))
print("Available evidence IDs:", sorted(available_evidence_ids))
print("Unknown evidence IDs:", sorted(unknown_evidence_ids))
print("Provenance valid:", provenance_valid)

Claimed evidence IDs: ['E001', 'E002', 'E003']
Available evidence IDs: ['E001', 'E002', 'E003', 'E004']
Unknown evidence IDs: []
Provenance valid: True


#### Preliminary Observation — Evidence Grounding

The neural component classified the candidate correspondence as
semantically valid with HIGH confidence. However, its justification
referred to attribute descriptions that are not present in the supplied
schema metadata.

This reveals a distinction between proposition correctness and evidence
grounding. A reconstructed proposition should not be admitted into the
design knowledge base solely because it is produced with high neural
confidence.

A candidate reconstructed proposition p should therefore be associated
with explicit evidence provenance e:

    Neural(E) -> (p, e)

and admitted as grounded knowledge only when the cited evidence can be
traced to the supplied evidence set:

    e ⊆ E

Evidence existence alone does not establish that the evidence is
sufficient to entail the proposition; evidence adequacy remains a
separate concern.

### 6.3 Symbolic Derivation of the L2 Finding

The neural component is not responsible for identifying the
normalization defect.

Given an accepted semantic correspondence between attributes, the
remaining reasoning is deterministic. Explicit uniqueness and
relationship metadata establish the dependency chain, while the
semantic correspondence establishes that the dependent property
materialized in the source model corresponds to the property determined
through the referenced entity.

For the current case:

    Unique(providers.provider_id)

    References(
        providers.organization_id,
        organizations.organization_id
    )

    Unique(organizations.organization_id)

    SemanticCorrespondence(
        providers.organization_state,
        organizations.organization_state
    )

These propositions provide the knowledge required to derive the
transitive dependency without asking the neural component to classify
the schema defect.

In [31]:
# Symbolically derive the L2 dependency chain for the current case

l2_facts = {
    "source_model": "stg_synthea__providers",
    "source_key": "provider_id",
    "source_fk": "organization_id",
    "target_model": "stg_synthea__organizations",
    "target_key": "organization_id",
    "source_dependent": "organization_state",
    "target_dependent": "organization_state",
    "source_key_unique": True,
    "target_key_unique": True,
    "relationship_declared": True,
    "semantic_correspondence": True
}

fd_1 = (
    l2_facts["source_key_unique"]
    and l2_facts["relationship_declared"]
)

fd_2 = (
    l2_facts["relationship_declared"]
    and l2_facts["target_key_unique"]
)

transitive_dependency = (
    fd_1
    and fd_2
    and l2_facts["semantic_correspondence"]
)

print("=== SYMBOLIC L2 DERIVATION ===\n")

print(
    "FD 1:",
    f'{l2_facts["source_key"]} -> {l2_facts["source_fk"]}',
    "=",
    fd_1
)

print(
    "FD 2:",
    f'{l2_facts["source_fk"]} -> {l2_facts["target_dependent"]}',
    "=",
    fd_2
)

print(
    "Semantic correspondence:",
    f'{l2_facts["source_dependent"]} <-> '
    f'{l2_facts["target_dependent"]}',
    "=",
    l2_facts["semantic_correspondence"]
)

print(
    "\nTransitive dependency:",
    f'{l2_facts["source_key"]} -> '
    f'{l2_facts["source_fk"]} -> '
    f'{l2_facts["source_dependent"]}',
    "=",
    transitive_dependency
)

=== SYMBOLIC L2 DERIVATION ===

FD 1: provider_id -> organization_id = True
FD 2: organization_id -> organization_state = True
Semantic correspondence: organization_state <-> organization_state = True

Transitive dependency: provider_id -> organization_id -> organization_state = True


In [32]:
# Deterministically verify the explicit structural facts used by L2

def column_by_name(model, column_name):
    for column in model.get("columns", []):
        if column.get("name") == column_name:
            return column
    return None


def has_unique_test(model, column_name):
    column = column_by_name(model, column_name)

    if column is None:
        return False

    return "unique" in (column.get("tests", []) or [])


def has_relationship_test(
    model,
    column_name,
    target_model,
    target_column
):
    column = column_by_name(model, column_name)

    if column is None:
        return False

    for test in column.get("tests", []) or []:
        if not isinstance(test, dict):
            continue

        relationship = test.get("relationships")

        if not relationship:
            continue

        target = relationship.get("to", "")
        field = relationship.get("field")

        if (
            target_model in target
            and field.lower() == target_column.lower()
        ):
            return True

    return False


source_key_unique = has_unique_test(
    source_model,
    "provider_id"
)

target_key_unique = has_unique_test(
    target_model,
    "organization_id"
)

relationship_declared = has_relationship_test(
    source_model,
    "organization_id",
    "stg_synthea__organizations",
    "organization_id"
)

print("=== EXPLICIT L2 FACTS ===\n")
print("Source key unique:", source_key_unique)
print("Target key unique:", target_key_unique)
print("Relationship declared:", relationship_declared)

=== EXPLICIT L2 FACTS ===

Source key unique: True
Target key unique: True
Relationship declared: True


In [33]:
# Complete the L2 neuro-symbolic derivation using extracted facts

semantic_correspondence = json.loads(
    v3_correspondence_response
)["semantic_correspondence"]

fd_source = (
    source_key_unique
    and relationship_declared
)

fd_target = (
    relationship_declared
    and target_key_unique
)

transitive_dependency = (
    fd_source
    and fd_target
    and semantic_correspondence
)

print("=== END-TO-END L2 DERIVATION ===\n")

print(
    "K_explicit:",
    source_key_unique,
    target_key_unique,
    relationship_declared
)

print(
    "K_reconstructed:",
    semantic_correspondence
)

print(
    "\nDerived FD:",
    "provider_id -> organization_id",
    "=",
    fd_source
)

print(
    "Derived FD:",
    "organization_id -> organization_state",
    "=",
    fd_target
)

print(
    "\nFinding:",
    "TRANSITIVE_DEPENDENCY",
    "=",
    transitive_dependency
)

=== END-TO-END L2 DERIVATION ===

K_explicit: True True True
K_reconstructed: True

Derived FD: provider_id -> organization_id = True
Derived FD: organization_id -> organization_state = True

Finding: TRANSITIVE_DEPENDENCY = True


## 7. L3 Neuro-Symbolic Derivation

The L3 experiment investigates whether a semantic relationship mismatch
can be established by separating semantic expectation reconstruction
from deterministic schema verification.

The neural component is not asked to identify a defect. Its role is
restricted to reconstructing the relationship that is semantically
expected for a source attribute.

The relationship actually declared in the schema is extracted
deterministically from the metadata.

The intended reasoning pattern is:

    Evidence
        -> ExpectedReference(source, expected_target)

    Schema Metadata
        -> DeclaredReference(source, declared_target)

    ExpectedReference(source, expected_target)
    ∧ DeclaredReference(source, declared_target)
    ∧ expected_target != declared_target
        -> SemanticRelationshipMismatch

In [34]:
# Extract the declared relationship for the selected L3 case

l3_metadata = extract_schema_metadata(v3_l3_audit_input)

l3_models = {
    model["name"]: model
    for model in l3_metadata.get("models", [])
}

l3_source_model = l3_models["dim_listings_hosts"]

host_id_column = column_by_name(
    l3_source_model,
    "HOST_ID"
)

declared_relationships = []

for test in host_id_column.get("tests", []) or []:
    if isinstance(test, dict) and "relationships" in test:
        relationship = test["relationships"]

        declared_relationships.append({
            "source_model": "dim_listings_hosts",
            "source_column": "HOST_ID",
            "target": relationship.get("to"),
            "target_column": relationship.get("field")
        })

print("=== DECLARED L3 RELATIONSHIP ===\n")

for relationship in declared_relationships:
    print("Source model:", relationship["source_model"])
    print("Source column:", relationship["source_column"])
    print("Declared target:", relationship["target"])
    print("Declared target column:", relationship["target_column"])

=== DECLARED L3 RELATIONSHIP ===

Source model: dim_listings_hosts
Source column: HOST_ID
Declared target: {{ ref('wow_agg_reviews') }}
Declared target column: date_sentiment_id


### 7.1 Expected Relationship Reconstruction

To prevent the neural component from directly evaluating the relationship
under test, the currently declared relationship for the source attribute
is removed from the evidence supplied during expectation reconstruction.

The neural task is therefore counterfactual with respect to the
relationship declaration:

    Given the remaining schema evidence, what entity and key should
    this attribute reference?

The original relationship stored in the benchmark ground truth remains
completely unavailable to the neural component.

In [35]:
# Build L3 evidence without the relationship currently declared for HOST_ID

import copy
import yaml

l3_reconstruction_metadata = copy.deepcopy(l3_metadata)

for model in l3_reconstruction_metadata.get("models", []):
    if model.get("name") != "dim_listings_hosts":
        continue

    for column in model.get("columns", []):
        if column.get("name", "").lower() != "host_id":
            continue

        filtered_tests = []

        for test in column.get("tests", []) or []:
            if isinstance(test, dict) and "relationships" in test:
                continue

            filtered_tests.append(test)

        column["tests"] = filtered_tests


l3_reconstruction_evidence = yaml.safe_dump(
    l3_reconstruction_metadata,
    sort_keys=False
)

print("=== L3 RECONSTRUCTION EVIDENCE ===\n")
print(l3_reconstruction_evidence)

=== L3 RECONSTRUCTION EVIDENCE ===

version: 2
models:
- name: daily_agg_reviews
  columns:
  - name: DATE_SENTIMENT_ID
    tests:
    - unique
    - not_null
  - name: REVIEW_SENTIMENT
    tests:
    - accepted_values:
        values:
        - positive
        - negative
        - neutral
        quote: true
- name: monthly_agg_reviews
  columns:
  - name: DATE_SENTIMENT_ID
    tests:
    - unique
    - not_null
  - name: REVIEW_SENTIMENT
    tests:
    - accepted_values:
        values:
        - positive
        - negative
        - neutral
        quote: true
- name: mom_agg_reviews
  columns:
  - name: DATE_SENTIMENT_ID
    tests:
    - unique
    - not_null
  - name: REVIEW_SENTIMENT
    tests:
    - accepted_values:
        values:
        - positive
        - negative
        - neutral
        quote: true
- name: wow_agg_reviews
  columns:
  - name: DATE_SENTIMENT_ID
    tests:
    - unique
    - not_null
  - name: REVIEW_SENTIMENT
    tests:
    - accepted_values:
        val

In [36]:
# V3 prompt — reconstruct the expected relationship for the L3 source attribute

V3_EXPECTED_REFERENCE_PROMPT = """You are analyzing a relational database schema.

Your task is NOT to audit the schema for defects.

Given the supplied schema metadata, determine which model and column
the specified source attribute is semantically expected to reference.

Use only the supplied schema evidence.

Do not identify schema defects.
Do not evaluate whether an existing relationship is correct or incorrect.
Do not invent entities or attributes that are not present in the metadata.

Return only valid JSON using this structure:

{
  "source_model": "<source model>",
  "source_column": "<source column>",
  "expected_target_model": "<target model>",
  "expected_target_column": "<target column>",
  "evidence": "<specific evidence supporting the expected relationship>",
  "confidence": "HIGH | MEDIUM | LOW"
}

If the available evidence is insufficient to establish an expected
relationship, return:

{
  "source_model": "<source model>",
  "source_column": "<source column>",
  "expected_target_model": null,
  "expected_target_column": null,
  "evidence": "<reason the relationship cannot be established>",
  "confidence": "LOW"
}

Do not include markdown or text outside the JSON object.
"""


l3_expected_reference_input = f"""
SCHEMA METADATA:

{l3_reconstruction_evidence}

SOURCE ATTRIBUTE:

model: dim_listings_hosts
column: HOST_ID
"""


v3_l3_expected_reference_response = invoke_v3_sonnet(
    V3_EXPECTED_REFERENCE_PROMPT,
    l3_expected_reference_input
)

print(v3_l3_expected_reference_response)

{
  "source_model": "dim_listings_hosts",
  "source_column": "HOST_ID",
  "expected_target_model": "dim_hosts",
  "expected_target_column": "HOST_ID",
  "evidence": "The dim_listings_hosts model contains a HOST_ID column, and there is a separate dim_hosts model with a HOST_ID column that is marked as unique and not null. This suggests that the HOST_ID in dim_listings_hosts is a foreign key reference to the HOST_ID primary key in the dim_hosts model.",
  "confidence": "HIGH"
}


### 7.2 Symbolic Verification of the Expected Relationship

The neural component reconstructs the relationship expected from the
available semantic evidence, while the relationship actually represented
in the schema is extracted independently from the schema metadata.

The final defect classification is deterministic:

    ExpectedReference(source, expected_target)
    ∧ DeclaredReference(source, declared_target)
    ∧ expected_target != declared_target
        -> SemanticRelationshipMismatch

The neural component does not perform this comparison and does not
classify the schema defect.

In [37]:
# Deterministically verify the reconstructed L3 relationship expectation

expected_reference = json.loads(
    v3_l3_expected_reference_response
)

expected_target_model = expected_reference[
    "expected_target_model"
]
expected_target_column = expected_reference[
    "expected_target_column"
]

declared_reference = declared_relationships[0]

declared_target_model = (
    declared_reference["target"]
    .replace("{{", "")
    .replace("}}", "")
    .replace("ref(", "")
    .replace(")", "")
    .replace("'", "")
    .replace('"', "")
    .strip()
)

declared_target_column = declared_reference[
    "target_column"
]

semantic_relationship_mismatch = (
    expected_target_model is not None
    and expected_target_column is not None
    and (
        expected_target_model.lower()
        != declared_target_model.lower()
        or expected_target_column.lower()
        != declared_target_column.lower()
    )
)

print("=== END-TO-END L3 DERIVATION ===\n")

print(
    "Expected reference:",
    f"dim_listings_hosts.HOST_ID -> "
    f"{expected_target_model}.{expected_target_column}"
)

print(
    "Declared reference:",
    f"dim_listings_hosts.HOST_ID -> "
    f"{declared_target_model}.{declared_target_column}"
)

print(
    "\nFinding:",
    "SEMANTIC_RELATIONSHIP_MISMATCH",
    "=",
    semantic_relationship_mismatch
)

=== END-TO-END L3 DERIVATION ===

Expected reference: dim_listings_hosts.HOST_ID -> dim_hosts.HOST_ID
Declared reference: dim_listings_hosts.HOST_ID -> wow_agg_reviews.date_sentiment_id

Finding: SEMANTIC_RELATIONSHIP_MISMATCH = True


## 8. Intermediate Schema Design Knowledge

The L1–L3 experiments indicate that neural reconstruction should not
directly produce schema defect classifications. Instead, neural and
deterministic processes produce typed propositions that can subsequently
be verified or combined through explicit reasoning rules.

We represent each proposition as:

    <type, arguments, provenance, derivation>

where:

- `type` identifies the proposition being asserted;
- `arguments` identify the schema elements involved;
- `provenance` records the evidence supporting the proposition;
- `derivation` records how the proposition was obtained.

Three derivation modes are initially distinguished:

    EXPLICIT
    NEURAL_RECONSTRUCTION
    SYMBOLIC_DERIVATION

This representation is intentionally minimal and is derived from the
requirements observed in the L1–L3 experiments. It is not intended to
define a complete schema knowledge language at this stage.

In [38]:
# Minimal representation for intermediate schema design knowledge

from dataclasses import dataclass, field
from typing import Dict, List, Any


@dataclass
class DesignProposition:
    proposition_type: str
    arguments: Dict[str, Any]
    provenance: List[Dict[str, Any]] = field(default_factory=list)
    derivation: str = "EXPLICIT"

    def __repr__(self):
        args = ", ".join(
            f"{key}={value}"
            for key, value in self.arguments.items()
        )

        return (
            f"{self.proposition_type}("
            f"{args}"
            f") [{self.derivation}]"
        )

In [39]:
# Test the intermediate representation with the L3 reconstructed expectation

l3_expected_reference_proposition = DesignProposition(
    proposition_type="EXPECTED_REFERENCE",
    arguments={
        "source_model": "dim_listings_hosts",
        "source_column": "HOST_ID",
        "target_model": expected_target_model,
        "target_column": expected_target_column
    },
    provenance=[
        {
            "source": "schema_metadata",
            "evidence": expected_reference["evidence"]
        }
    ],
    derivation="NEURAL_RECONSTRUCTION"
)

print(l3_expected_reference_proposition)
print()
print(l3_expected_reference_proposition.provenance)

EXPECTED_REFERENCE(source_model=dim_listings_hosts, source_column=HOST_ID, target_model=dim_hosts, target_column=HOST_ID) [NEURAL_RECONSTRUCTION]

[{'source': 'schema_metadata', 'evidence': 'The dim_listings_hosts model contains a HOST_ID column, and there is a separate dim_hosts model with a HOST_ID column that is marked as unique and not null. This suggests that the HOST_ID in dim_listings_hosts is a foreign key reference to the HOST_ID primary key in the dim_hosts model.'}]


In [40]:
# Represent the explicitly declared L3 relationship

l3_declared_reference_proposition = DesignProposition(
    proposition_type="DECLARED_REFERENCE",
    arguments={
        "source_model": declared_reference["source_model"],
        "source_column": declared_reference["source_column"],
        "target_model": declared_target_model,
        "target_column": declared_target_column
    },
    provenance=[
        {
            "source": "schema_metadata",
            "evidence": "relationships test"
        }
    ],
    derivation="EXPLICIT"
)

print(l3_declared_reference_proposition)

DECLARED_REFERENCE(source_model=dim_listings_hosts, source_column=HOST_ID, target_model=wow_agg_reviews, target_column=date_sentiment_id) [EXPLICIT]


In [41]:
# Symbolic rule over intermediate design propositions

def derive_semantic_relationship_mismatch(
    expected_reference,
    declared_reference
):
    if expected_reference.proposition_type != "EXPECTED_REFERENCE":
        return None

    if declared_reference.proposition_type != "DECLARED_REFERENCE":
        return None

    expected = expected_reference.arguments
    declared = declared_reference.arguments

    same_source = (
        expected["source_model"].lower()
        == declared["source_model"].lower()
        and
        expected["source_column"].lower()
        == declared["source_column"].lower()
    )

    same_target = (
        expected["target_model"].lower()
        == declared["target_model"].lower()
        and
        expected["target_column"].lower()
        == declared["target_column"].lower()
    )

    if not same_source or same_target:
        return None

    return DesignProposition(
        proposition_type="SEMANTIC_RELATIONSHIP_MISMATCH",
        arguments={
            "source_model": expected["source_model"],
            "source_column": expected["source_column"],
            "expected_target_model": expected["target_model"],
            "expected_target_column": expected["target_column"],
            "declared_target_model": declared["target_model"],
            "declared_target_column": declared["target_column"]
        },
        provenance=[
            {
                "source": "derived_from",
                "propositions": [
                    repr(expected_reference),
                    repr(declared_reference)
                ]
            }
        ],
        derivation="SYMBOLIC_DERIVATION"
    )


l3_finding_proposition = derive_semantic_relationship_mismatch(
    l3_expected_reference_proposition,
    l3_declared_reference_proposition
)

print("=== L3 KNOWLEDGE DERIVATION ===\n")
print(l3_expected_reference_proposition)
print(l3_declared_reference_proposition)
print()
print(l3_finding_proposition)

=== L3 KNOWLEDGE DERIVATION ===

EXPECTED_REFERENCE(source_model=dim_listings_hosts, source_column=HOST_ID, target_model=dim_hosts, target_column=HOST_ID) [NEURAL_RECONSTRUCTION]
DECLARED_REFERENCE(source_model=dim_listings_hosts, source_column=HOST_ID, target_model=wow_agg_reviews, target_column=date_sentiment_id) [EXPLICIT]

SEMANTIC_RELATIONSHIP_MISMATCH(source_model=dim_listings_hosts, source_column=HOST_ID, expected_target_model=dim_hosts, expected_target_column=HOST_ID, declared_target_model=wow_agg_reviews, declared_target_column=date_sentiment_id) [SYMBOLIC_DERIVATION]


In [42]:
# Represent the L1 reconstructed and explicit uniqueness knowledge

l1_expected_unique_proposition = DesignProposition(
    proposition_type="EXPECTED_UNIQUE",
    arguments={
        "model": "int_lever__requisition_users",
        "columns": ["requisition_id"]
    },
    provenance=[
        {
            "source": "schema_evidence",
            "evidence": (
                "The attribute requisition_id is described as "
                "'Unique id of the requisition.'"
            )
        }
    ],
    derivation="NEURAL_RECONSTRUCTION"
)


l1_declared_unique_propositions = [
    DesignProposition(
        proposition_type="DECLARED_UNIQUE",
        arguments={
            "model": q["model"],
            "columns": q["columns"]
        },
        provenance=[
            {
                "source": "schema_metadata",
                "evidence": "explicit uniqueness test"
            }
        ],
        derivation="EXPLICIT"
    )
    for q in q_explicit
]


print("=== L1 KNOWLEDGE ===\n")

print("Reconstructed:")
print(l1_expected_unique_proposition)

print("\nExplicit uniqueness propositions:")
for proposition in l1_declared_unique_propositions:
    print(proposition)

=== L1 KNOWLEDGE ===

Reconstructed:
EXPECTED_UNIQUE(model=int_lever__requisition_users, columns=['requisition_id']) [NEURAL_RECONSTRUCTION]

Explicit uniqueness propositions:
DECLARED_UNIQUE(model=int_lever__contact_info, columns=('contact_id',)) [EXPLICIT]
DECLARED_UNIQUE(model=int_lever__interview_feedback, columns=('feedback_form_id', 'interview_id')) [EXPLICIT]
DECLARED_UNIQUE(model=int_lever__opportunity_application, columns=('opportunity_id',)) [EXPLICIT]
DECLARED_UNIQUE(model=int_lever__opportunity_contact_info, columns=('opportunity_id',)) [EXPLICIT]
DECLARED_UNIQUE(model=int_lever__opportunity_tags, columns=('opportunity_id',)) [EXPLICIT]
DECLARED_UNIQUE(model=int_lever__opportunity_users, columns=('opportunity_id',)) [EXPLICIT]
DECLARED_UNIQUE(model=int_lever__posting_applications, columns=('posting_id',)) [EXPLICIT]
DECLARED_UNIQUE(model=int_lever__posting_interviews, columns=('posting_id',)) [EXPLICIT]
DECLARED_UNIQUE(model=int_lever__posting_tags, columns=('posting_id',))

In [43]:
# Symbolic rule for deriving a missing uniqueness constraint

def normalize_columns(columns):
    return tuple(sorted(
        column.lower()
        for column in columns
    ))


def derive_missing_unique(
    expected_unique,
    knowledge_base
):
    if expected_unique.proposition_type != "EXPECTED_UNIQUE":
        return None

    expected_model = expected_unique.arguments["model"].lower()
    expected_columns = normalize_columns(
        expected_unique.arguments["columns"]
    )

    for proposition in knowledge_base:

        if proposition.proposition_type != "DECLARED_UNIQUE":
            continue

        declared_model = proposition.arguments["model"].lower()
        declared_columns = normalize_columns(
            proposition.arguments["columns"]
        )

        if (
            expected_model == declared_model
            and expected_columns == declared_columns
        ):
            return None

    return DesignProposition(
        proposition_type="MISSING_UNIQUE",
        arguments={
            "model": expected_unique.arguments["model"],
            "columns": list(expected_columns)
        },
        provenance=[
            {
                "source": "derived_from",
                "propositions": [
                    repr(expected_unique)
                ],
                "absence_check": "No matching DECLARED_UNIQUE proposition"
            }
        ],
        derivation="SYMBOLIC_DERIVATION"
    )


l1_knowledge_base = (
    [l1_expected_unique_proposition]
    + l1_declared_unique_propositions
)

l1_finding_proposition = derive_missing_unique(
    l1_expected_unique_proposition,
    l1_knowledge_base
)

print("=== L1 KNOWLEDGE DERIVATION ===\n")

print(l1_expected_unique_proposition)

print("\nMatching DECLARED_UNIQUE:")
matching_declared = [
    p
    for p in l1_declared_unique_propositions
    if (
        p.arguments["model"].lower()
        == l1_expected_unique_proposition.arguments["model"].lower()
        and
        normalize_columns(p.arguments["columns"])
        == normalize_columns(
            l1_expected_unique_proposition.arguments["columns"]
        )
    )
]

print(matching_declared)

print("\nDerived finding:")
print(l1_finding_proposition)

=== L1 KNOWLEDGE DERIVATION ===

EXPECTED_UNIQUE(model=int_lever__requisition_users, columns=['requisition_id']) [NEURAL_RECONSTRUCTION]

Matching DECLARED_UNIQUE:
[]

Derived finding:
MISSING_UNIQUE(model=int_lever__requisition_users, columns=['requisition_id']) [SYMBOLIC_DERIVATION]


### 8.1 L2 Representation Using Design Propositions

The L2 experiment is now expressed using the same intermediate
representation used for L1 and L3.

Explicit structural facts are represented as propositions derived from
schema metadata, while semantic correspondence is represented as a
neural reconstruction.

Functional dependencies and the final transitive dependency are then
represented as symbolically derived propositions.

In [44]:
# Represent the L2 knowledge using DesignProposition

l2_source_unique = DesignProposition(
    proposition_type="UNIQUE",
    arguments={
        "model": "stg_synthea__providers",
        "columns": ["provider_id"]
    },
    provenance=[
        {
            "source": "schema_metadata",
            "evidence": "unique test"
        }
    ],
    derivation="EXPLICIT"
)


l2_relationship = DesignProposition(
    proposition_type="REFERENCE",
    arguments={
        "source_model": "stg_synthea__providers",
        "source_column": "organization_id",
        "target_model": "stg_synthea__organizations",
        "target_column": "organization_id"
    },
    provenance=[
        {
            "source": "schema_metadata",
            "evidence": "relationships test"
        }
    ],
    derivation="EXPLICIT"
)


l2_target_unique = DesignProposition(
    proposition_type="UNIQUE",
    arguments={
        "model": "stg_synthea__organizations",
        "columns": ["organization_id"]
    },
    provenance=[
        {
            "source": "schema_metadata",
            "evidence": "unique test"
        }
    ],
    derivation="EXPLICIT"
)


l2_correspondence_result = json.loads(
    v3_correspondence_response
)


l2_semantic_correspondence = DesignProposition(
    proposition_type="SEMANTIC_CORRESPONDENCE",
    arguments={
        "source_model": "stg_synthea__providers",
        "source_column": "organization_state",
        "target_model": "stg_synthea__organizations",
        "target_column": "organization_state"
    },
    provenance=[
        {
            "source": "schema_evidence",
            "evidence_ids": l2_correspondence_result.get(
                "evidence_ids",
                []
            )
        }
    ],
    derivation="NEURAL_RECONSTRUCTION"
)

print("=== L2 DESIGN KNOWLEDGE ===\n")

for proposition in [
    l2_source_unique,
    l2_relationship,
    l2_target_unique,
    l2_semantic_correspondence
]:
    print(proposition)

=== L2 DESIGN KNOWLEDGE ===

UNIQUE(model=stg_synthea__providers, columns=['provider_id']) [EXPLICIT]
REFERENCE(source_model=stg_synthea__providers, source_column=organization_id, target_model=stg_synthea__organizations, target_column=organization_id) [EXPLICIT]
UNIQUE(model=stg_synthea__organizations, columns=['organization_id']) [EXPLICIT]
SEMANTIC_CORRESPONDENCE(source_model=stg_synthea__providers, source_column=organization_state, target_model=stg_synthea__organizations, target_column=organization_state) [NEURAL_RECONSTRUCTION]


In [45]:
# Symbolically derive the functional dependencies for L2

def derive_l2_functional_dependencies(
    source_unique,
    relationship,
    target_unique
):
    if source_unique.proposition_type != "UNIQUE":
        return []

    if relationship.proposition_type != "REFERENCE":
        return []

    if target_unique.proposition_type != "UNIQUE":
        return []

    source_key = source_unique.arguments["columns"]

    if len(source_key) != 1:
        return []

    source_key = source_key[0]

    # The source key functionally determines the source foreign key.
    fd_source = DesignProposition(
        proposition_type="FUNCTIONAL_DEPENDENCY",
        arguments={
            "determinant_model": source_unique.arguments["model"],
            "determinant_column": source_key,
            "dependent_model": relationship.arguments["source_model"],
            "dependent_column": relationship.arguments["source_column"]
        },
        provenance=[
            {
                "source": "derived_from",
                "propositions": [
                    repr(source_unique),
                    repr(relationship)
                ]
            }
        ],
        derivation="SYMBOLIC_DERIVATION"
    )

    target_columns = normalize_columns(
        target_unique.arguments["columns"]
    )

    relationship_target = (
        relationship.arguments["target_column"].lower()
    )

    if relationship_target not in target_columns:
        return [fd_source]

    # The foreign key identifies a unique target tuple.
    # The semantically corresponding dependent attribute will be
    # introduced in the next inference step.
    fd_reference = DesignProposition(
        proposition_type="REFERENCE_TO_UNIQUE",
        arguments={
            "source_model": relationship.arguments["source_model"],
            "source_column": relationship.arguments["source_column"],
            "target_model": relationship.arguments["target_model"],
            "target_column": relationship.arguments["target_column"]
        },
        provenance=[
            {
                "source": "derived_from",
                "propositions": [
                    repr(relationship),
                    repr(target_unique)
                ]
            }
        ],
        derivation="SYMBOLIC_DERIVATION"
    )

    return [fd_source, fd_reference]


l2_derived_structural = derive_l2_functional_dependencies(
    l2_source_unique,
    l2_relationship,
    l2_target_unique
)

print("=== L2 SYMBOLIC STRUCTURAL DERIVATION ===\n")

for proposition in l2_derived_structural:
    print(proposition)

=== L2 SYMBOLIC STRUCTURAL DERIVATION ===

FUNCTIONAL_DEPENDENCY(determinant_model=stg_synthea__providers, determinant_column=provider_id, dependent_model=stg_synthea__providers, dependent_column=organization_id) [SYMBOLIC_DERIVATION]
REFERENCE_TO_UNIQUE(source_model=stg_synthea__providers, source_column=organization_id, target_model=stg_synthea__organizations, target_column=organization_id) [SYMBOLIC_DERIVATION]


In [46]:
# Combine structural and semantic knowledge to derive the L2 dependent property

def derive_semantic_functional_dependency(
    reference_to_unique,
    semantic_correspondence
):
    if reference_to_unique.proposition_type != "REFERENCE_TO_UNIQUE":
        return None

    if semantic_correspondence.proposition_type != "SEMANTIC_CORRESPONDENCE":
        return None

    reference = reference_to_unique.arguments
    correspondence = semantic_correspondence.arguments

    same_source_model = (
        reference["source_model"].lower()
        == correspondence["source_model"].lower()
    )

    same_target_model = (
        reference["target_model"].lower()
        == correspondence["target_model"].lower()
    )

    if not same_source_model or not same_target_model:
        return None

    return DesignProposition(
        proposition_type="FUNCTIONAL_DEPENDENCY",
        arguments={
            "determinant_model": reference["source_model"],
            "determinant_column": reference["source_column"],
            "dependent_model": correspondence["source_model"],
            "dependent_column": correspondence["source_column"]
        },
        provenance=[
            {
                "source": "derived_from",
                "propositions": [
                    repr(reference_to_unique),
                    repr(semantic_correspondence)
                ]
            }
        ],
        derivation="SYMBOLIC_DERIVATION"
    )


l2_semantic_fd = derive_semantic_functional_dependency(
    l2_derived_structural[1],
    l2_semantic_correspondence
)

print("=== L2 SEMANTIC FD DERIVATION ===\n")
print(l2_semantic_fd)

=== L2 SEMANTIC FD DERIVATION ===

FUNCTIONAL_DEPENDENCY(determinant_model=stg_synthea__providers, determinant_column=organization_id, dependent_model=stg_synthea__providers, dependent_column=organization_state) [SYMBOLIC_DERIVATION]


In [47]:
# Symbolically derive the final L2 transitive dependency

def derive_transitive_dependency(fd_1, fd_2):
    if fd_1.proposition_type != "FUNCTIONAL_DEPENDENCY":
        return None

    if fd_2.proposition_type != "FUNCTIONAL_DEPENDENCY":
        return None

    first = fd_1.arguments
    second = fd_2.arguments

    same_model = (
        first["dependent_model"].lower()
        == second["determinant_model"].lower()
    )

    chained_attribute = (
        first["dependent_column"].lower()
        == second["determinant_column"].lower()
    )

    if not same_model or not chained_attribute:
        return None

    return DesignProposition(
        proposition_type="TRANSITIVE_DEPENDENCY",
        arguments={
            "model": first["determinant_model"],
            "determinant_column": first["determinant_column"],
            "intermediate_column": first["dependent_column"],
            "dependent_column": second["dependent_column"]
        },
        provenance=[
            {
                "source": "derived_from",
                "propositions": [
                    repr(fd_1),
                    repr(fd_2)
                ]
            }
        ],
        derivation="SYMBOLIC_DERIVATION"
    )


l2_finding_proposition = derive_transitive_dependency(
    l2_derived_structural[0],
    l2_semantic_fd
)

print("=== L2 KNOWLEDGE DERIVATION ===\n")

print("FD 1:")
print(l2_derived_structural[0])

print("\nFD 2:")
print(l2_semantic_fd)

print("\nDerived finding:")
print(l2_finding_proposition)

=== L2 KNOWLEDGE DERIVATION ===

FD 1:
FUNCTIONAL_DEPENDENCY(determinant_model=stg_synthea__providers, determinant_column=provider_id, dependent_model=stg_synthea__providers, dependent_column=organization_id) [SYMBOLIC_DERIVATION]

FD 2:
FUNCTIONAL_DEPENDENCY(determinant_model=stg_synthea__providers, determinant_column=organization_id, dependent_model=stg_synthea__providers, dependent_column=organization_state) [SYMBOLIC_DERIVATION]

Derived finding:
TRANSITIVE_DEPENDENCY(model=stg_synthea__providers, determinant_column=provider_id, intermediate_column=organization_id, dependent_column=organization_state) [SYMBOLIC_DERIVATION]


## 8. V3 Batch Evaluation

The exploratory experiments above define the neuro-symbolic auditing
procedure for the three evidence levels. We now apply the same procedures
systematically to the RDBScore benchmark instances.

In [48]:
# Robust JSON parser for V3 neural responses

def parse_v3_json_response(response):

    if response is None:
        raise ValueError("Empty model response")

    text = response.strip()

    if not text:
        raise ValueError("Empty model response")

    # Remove Markdown code fences when present
    if text.startswith("```"):
        lines = text.splitlines()

        if lines and lines[0].startswith("```"):
            lines = lines[1:]

        if lines and lines[-1].strip() == "```":
            lines = lines[:-1]

        text = "\n".join(lines).strip()

    # First try: response is already valid JSON
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    # Second try: extract the first complete JSON object
    decoder = json.JSONDecoder()

    start = text.find("{")

    if start == -1:
        raise ValueError(
            f"No JSON object found in model response: {text[:300]!r}"
        )

    try:
        obj, _ = decoder.raw_decode(text[start:])
        return obj
    except json.JSONDecodeError as e:
        raise ValueError(
            f"Invalid JSON model response: {text[:500]!r}"
        ) from e

In [49]:
### V3 batch audit — L1_REMOVE_UNIQUE

def audit_v3_remove_unique(audit_input):

    # 1. Neural reconstruction of expected uniqueness
    response = invoke_v3_sonnet(
        system_prompt=V3_EXPECTATION_PROMPT,
        evidence=audit_input,
        max_tokens=2000,
    )

    result = parse_v3_json_response(response)
    q_expected = result.get("expectations", [])

    # 2. Deterministic extraction of declared uniqueness
    q_explicit = extract_explicit_unique_expectations(
        audit_input
    )

    # 3. Deterministic comparison
    explicit_keys = {
        expectation_key(q)
        for q in q_explicit
    }

    q_missing = [
        q
        for q in q_expected
        if expectation_key(q) not in explicit_keys
    ]

    # 4. Produce schema findings
    findings = []

    for q in q_missing:
        findings.append({
            "finding": "MISSING_UNIQUE",
            "model": q["model"],
            "columns": q["columns"],
            "evidence": q.get("evidence"),
            "confidence": q.get("confidence"),
        })

    return {
        "findings": findings,
        "reconstructed_expectations": q_expected,
        "explicit_expectations": q_explicit,
        "raw_response": response,
    }

In [50]:
# Sanity check — reproduce the exploratory L1 result

l1_batch_test = audit_v3_remove_unique(
    v3_audit_input
)

print("=== V3 L1 BATCH SANITY CHECK ===\n")

print(
    "Reconstructed expectations:",
    len(l1_batch_test["reconstructed_expectations"])
)

print(
    "Explicit expectations:",
    len(l1_batch_test["explicit_expectations"])
)

print(
    "Findings:",
    len(l1_batch_test["findings"])
)

print("\n=== FINDINGS ===")

for finding in l1_batch_test["findings"]:
    print(json.dumps(finding, indent=2))

=== V3 L1 BATCH SANITY CHECK ===

Reconstructed expectations: 10
Explicit expectations: 9
Findings: 1

=== FINDINGS ===
{
  "finding": "MISSING_UNIQUE",
  "model": "int_lever__requisition_users",
  "columns": [
    "requisition_id"
  ],
  "evidence": "Column 'requisition_id' has a 'not_null' test defined in the schema metadata.",
  "confidence": "MEDIUM"
}


In [51]:
# Batch execution — L1_REMOVE_UNIQUE

from pathlib import Path
import time

L1_BATCH_DIR = Path(
    "rdbscore_generated_v2/L1/L1_REMOVE_UNIQUE"
)

L1_V3_RESULTS_PATH = Path(
    "v3_l1_remove_unique_results.jsonl"
)

l1_cases = sorted([
    p for p in L1_BATCH_DIR.iterdir()
    if p.is_dir() and (p / "audit_input.txt").exists()
])

print("L1_REMOVE_UNIQUE instances:", len(l1_cases))
print("Results file:", L1_V3_RESULTS_PATH)

L1_REMOVE_UNIQUE instances: 300
Results file: v3_l1_remove_unique_results.jsonl


In [52]:
# Execute V3 batch — L1_REMOVE_UNIQUE

completed_cases = set()

if L1_V3_RESULTS_PATH.exists():
    with L1_V3_RESULTS_PATH.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                record = json.loads(line)
                if record.get("status") == "OK":
                    completed_cases.add(record["case_id"])

print("Already completed:", len(completed_cases))
print("Remaining:", len(l1_cases) - len(completed_cases))


for i, case_path in enumerate(l1_cases, start=1):

    case_id = case_path.name

    if case_id in completed_cases:
        continue

    audit_input_path = case_path / "audit_input.txt"

    try:
        audit_input = audit_input_path.read_text(
            encoding="utf-8"
        )

        result = audit_v3_remove_unique(
            audit_input
        )

        record = {
            "case_id": case_id,
            "operator_id": "L1_REMOVE_UNIQUE",
            "status": "OK",
            "findings": result["findings"],
            "reconstructed_expectations":
                result["reconstructed_expectations"],
            "explicit_expectations":
                result["explicit_expectations"]
        }

    except Exception as e:

        record = {
            "case_id": case_id,
            "operator_id": "L1_REMOVE_UNIQUE",
            "status": "ERROR",
            "error": str(e)
        }

    with L1_V3_RESULTS_PATH.open(
        "a",
        encoding="utf-8"
    ) as f:
        f.write(
            json.dumps(
                record,
                ensure_ascii=False
            ) + "\n"
        )

    if record["status"] == "OK":
        completed_cases.add(case_id)

    print(
        f"[{i:03d}/{len(l1_cases)}] "
        f"{case_id} "
        f"{record['status']} "
        f"findings={len(record.get('findings', []))}"
    )

Already completed: 300
Remaining: 0


In [53]:
# Consolidate L1 batch results — keep the latest successful result per case

latest_ok_by_case = {}

with L1_V3_RESULTS_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue

        record = json.loads(line)

        if record.get("status") == "OK":
            latest_ok_by_case[record["case_id"]] = record


print("Unique successful cases:", len(latest_ok_by_case))

finding_counts = Counter(
    len(record.get("findings", []))
    for record in latest_ok_by_case.values()
)

print("\n=== FINDINGS PER CASE ===")

for n_findings, count in sorted(finding_counts.items()):
    print(f"{n_findings} findings: {count} cases")

Unique successful cases: 300

=== FINDINGS PER CASE ===
0 findings: 150 cases
1 findings: 99 cases
2 findings: 36 cases
3 findings: 8 cases
4 findings: 7 cases


In [54]:
# Inspect one L1 ground-truth record before defining the evaluator

sample_case_id = sorted(latest_ok_by_case.keys())[0]
sample_case_path = L1_BATCH_DIR / sample_case_id

ground_truth_files = sorted(
    sample_case_path.glob("*ground*truth*.json")
)

print("Case:", sample_case_id)
print("Ground-truth files:", ground_truth_files)

if ground_truth_files:
    sample_gt = json.loads(
        ground_truth_files[0].read_text(encoding="utf-8")
    )

    print("\n=== GROUND TRUTH ===")
    print(json.dumps(sample_gt, indent=2, ensure_ascii=False))

Case: L1_REMOVE_UNIQUE_0088503f05f3
Ground-truth files: [PosixPath('rdbscore_generated_v2/L1/L1_REMOVE_UNIQUE/L1_REMOVE_UNIQUE_0088503f05f3/ground_truth.json')]

=== GROUND TRUTH ===
{
  "mutation_id": "L1_REMOVE_UNIQUE_0088503f05f3",
  "operator_id": "L1_REMOVE_UNIQUE",
  "level": "L1",
  "defect_family": "Constraint Defects",
  "mutation_type": "REMOVE_UNIQUE",
  "expected_defect": "Missing explicitly declared uniqueness constraint/test.",
  "evidence_requirement": "EXPLICIT_METADATA",
  "benchmark": "Spider2-DBT",
  "protocol_version": "RDBSCORE_V2",
  "project": "lever001",
  "artifact": "models/intermediate/int_lever.yml",
  "candidate": {
    "model": "int_lever__requisition_users",
    "column": "requisition_id"
  },
  "evidence": null
}


In [55]:
# Evaluate V3 — L1_REMOVE_UNIQUE against ground truth

l1_evaluation = []

for case_id, record in latest_ok_by_case.items():

    case_path = L1_BATCH_DIR / case_id

    gt = json.loads(
        (case_path / "ground_truth.json").read_text(
            encoding="utf-8"
        )
    )

    target_model = gt["candidate"]["model"]
    target_column = gt["candidate"]["column"]

    findings = record.get("findings", [])

    matching_findings = [
        finding
        for finding in findings
        if (
            finding.get("finding") == "MISSING_UNIQUE"
            and finding.get("model") == target_model
            and target_column in finding.get("columns", [])
        )
    ]

    target_detected = len(matching_findings) > 0

    exact_case_match = (
        target_detected
        and len(findings) == 1
    )

    l1_evaluation.append({
        "case_id": case_id,
        "target_model": target_model,
        "target_column": target_column,
        "n_findings": len(findings),
        "target_detected": target_detected,
        "exact_case_match": exact_case_match,
    })


n = len(l1_evaluation)

detected = sum(
    r["target_detected"]
    for r in l1_evaluation
)

exact = sum(
    r["exact_case_match"]
    for r in l1_evaluation
)

not_detected = n - detected


print("=== V3 L1_REMOVE_UNIQUE EVALUATION ===\n")

print("N:", n)

print(
    f"Target detected: {detected} "
    f"({detected / n:.2%})"
)

print(
    f"Exact case match: {exact} "
    f"({exact / n:.2%})"
)

print(
    f"Target not detected: {not_detected} "
    f"({not_detected / n:.2%})"
)

=== V3 L1_REMOVE_UNIQUE EVALUATION ===

N: 300
Target detected: 126 (42.00%)
Exact case match: 81 (27.00%)
Target not detected: 174 (58.00%)


In [56]:
# L2 batch support — extract relationship structures and candidate correspondences

def extract_l2_relationship_candidates(audit_input):

    metadata = extract_schema_metadata(audit_input)

    models = metadata.get("models", [])

    models_by_name = {
        model["name"]: model
        for model in models
        if model.get("name")
    }

    structures = []

    for source_model in models:

        source_model_name = source_model.get("name")

        for source_column in source_model.get("columns", []):

            source_column_name = source_column.get("name")

            for test in source_column.get("tests", []) or []:

                if not isinstance(test, dict):
                    continue

                relationship = test.get("relationships")

                if not relationship:
                    continue

                target = relationship.get("to", "")
                target_column = relationship.get("field")

                if not target_column:
                    continue

                # Resolve the referenced model from the models
                # available in the evidence package.
                target_model_name = next(
                    (
                        name
                        for name in models_by_name
                        if name in target
                    ),
                    None
                )

                if target_model_name is None:
                    continue

                target_model = models_by_name[target_model_name]

                source_columns = {
                    column.get("name")
                    for column in source_model.get("columns", [])
                    if column.get("name")
                }

                target_columns = {
                    column.get("name")
                    for column in target_model.get("columns", [])
                    if column.get("name")
                }

                shared_columns = sorted(
                    source_columns & target_columns
                )

                candidate_correspondences = [
                    {
                        "source_model": source_model_name,
                        "source_column": column,
                        "target_model": target_model_name,
                        "target_column": column
                    }
                    for column in shared_columns
                    if column.lower() != source_column_name.lower()
                ]

                structures.append({
                    "source_model": source_model_name,
                    "relationship_column": source_column_name,
                    "target_model": target_model_name,
                    "target_column": target_column,
                    "source_model_object": source_model,
                    "target_model_object": target_model,
                    "candidate_correspondences":
                        candidate_correspondences
                })

    return structures

In [57]:
# Sanity check — automatic L2 relationship extraction

l2_structures = extract_l2_relationship_candidates(
    v3_l2_audit_input
)

print("Relationship structures:", len(l2_structures))

for structure in l2_structures:
    print(
        "\n",
        structure["source_model"],
        ".",
        structure["relationship_column"],
        " -> ",
        structure["target_model"],
        ".",
        structure["target_column"],
        sep=""
    )

    for candidate in structure["candidate_correspondences"]:
        print(
            "  Candidate:",
            f'{candidate["source_model"]}.'
            f'{candidate["source_column"]}',
            "<->",
            f'{candidate["target_model"]}.'
            f'{candidate["target_column"]}'
        )

Relationship structures: 43

stg_synthea__allergies.patient_id -> stg_synthea__patients.patient_id

stg_synthea__allergies.encounter_id -> stg_synthea__encounters.encounter_id
  Candidate: stg_synthea__allergies.patient_id <-> stg_synthea__encounters.patient_id

stg_synthea__careplans.patient_id -> stg_synthea__patients.patient_id

stg_synthea__careplans.encounter_id -> stg_synthea__encounters.encounter_id
  Candidate: stg_synthea__careplans.patient_id <-> stg_synthea__encounters.patient_id

stg_synthea__claims_transactions.claim_id -> stg_synthea__claims.claim_id
  Candidate: stg_synthea__claims_transactions.department_id <-> stg_synthea__claims.department_id
  Candidate: stg_synthea__claims_transactions.encounter_id <-> stg_synthea__claims.encounter_id
  Candidate: stg_synthea__claims_transactions.patient_id <-> stg_synthea__claims.patient_id
  Candidate: stg_synthea__claims_transactions.provider_id <-> stg_synthea__claims.provider_id
  Candidate: stg_synthea__claims_transactions.sup

In [58]:
# Filter L2 structures by the explicit premises required for derivation

def filter_l2_structural_candidates(structures):

    filtered = []

    for structure in structures:

        source_model = structure["source_model_object"]
        target_model = structure["target_model_object"]

        relationship_column = structure["relationship_column"]
        target_column = structure["target_column"]

        # Candidate source keys are UNIQUE columns in the source model,
        # excluding the relationship column itself.
        source_unique_columns = [
            column.get("name")
            for column in source_model.get("columns", [])
            if (
                column.get("name")
                and column.get("name") != relationship_column
                and has_unique_test(
                    source_model,
                    column.get("name")
                )
            )
        ]

        target_key_unique = has_unique_test(
            target_model,
            target_column
        )

        if (
            source_unique_columns
            and target_key_unique
            and structure["candidate_correspondences"]
        ):
            filtered.append({
                **structure,
                "source_unique_columns":
                    source_unique_columns,
                "target_key_unique":
                    target_key_unique
            })

    return filtered


l2_structural_candidates = filter_l2_structural_candidates(
    l2_structures
)

print(
    "Structurally eligible L2 relationships:",
    len(l2_structural_candidates)
)

for structure in l2_structural_candidates:

    print(
        "\nRelationship:",
        f'{structure["source_model"]}.'
        f'{structure["relationship_column"]}',
        "->",
        f'{structure["target_model"]}.'
        f'{structure["target_column"]}'
    )

    print(
        "Source unique columns:",
        structure["source_unique_columns"]
    )

    for candidate in structure["candidate_correspondences"]:
        print(
            "  Candidate:",
            f'{candidate["source_model"]}.'
            f'{candidate["source_column"]}',
            "<->",
            f'{candidate["target_model"]}.'
            f'{candidate["target_column"]}'
        )

Structurally eligible L2 relationships: 10

Relationship: stg_synthea__careplans.encounter_id -> stg_synthea__encounters.encounter_id
Source unique columns: ['careplan_id']
  Candidate: stg_synthea__careplans.patient_id <-> stg_synthea__encounters.patient_id

Relationship: stg_synthea__claims_transactions.claim_id -> stg_synthea__claims.claim_id
Source unique columns: ['claim_transaction_id']
  Candidate: stg_synthea__claims_transactions.department_id <-> stg_synthea__claims.department_id
  Candidate: stg_synthea__claims_transactions.encounter_id <-> stg_synthea__claims.encounter_id
  Candidate: stg_synthea__claims_transactions.patient_id <-> stg_synthea__claims.patient_id
  Candidate: stg_synthea__claims_transactions.provider_id <-> stg_synthea__claims.provider_id
  Candidate: stg_synthea__claims_transactions.supervising_provider_id <-> stg_synthea__claims.supervising_provider_id

Relationship: stg_synthea__claims_transactions.encounter_id -> stg_synthea__encounters.encounter_id
Sourc

In [59]:
# V3 batch prompt — validate L2 semantic correspondences

V3_L2_BATCH_CORRESPONDENCE_PROMPT = """
You are analyzing relational schema evidence.

Your task is NOT to audit the schema for defects.

You will receive candidate correspondences between attributes
and identified evidence units for each candidate.

For each candidate, determine whether the two attributes represent
the same conceptual property in the domain.

Use only the supplied evidence units.

If a semantic correspondence exists, identify the evidence units
supporting the conclusion using their evidence IDs.

Do not invent descriptions, documentation, relationships,
constraints, or domain information that is not present in the
supplied evidence.

Do not evaluate normalization.
Do not identify functional dependencies.
Do not identify transitive dependencies.
Do not report schema defects.

Return only valid JSON using this structure:

{
  "correspondences": [
    {
      "candidate_id": "C001",
      "semantic_correspondence": true,
      "evidence_ids": ["E001", "E002"],
      "confidence": "HIGH"
    }
  ]
}

Return exactly one result for every supplied candidate_id.

Do not include markdown or text outside the JSON object.
"""

In [60]:
# Build addressable L2 candidate evidence for batch auditing

def build_l2_candidate_payload(structures):

    candidates = []
    evidence_units = []

    candidate_counter = 1
    evidence_counter = 1

    for structure in structures:

        source_model = structure["source_model"]
        target_model = structure["target_model"]

        relationship_column = structure["relationship_column"]
        target_column = structure["target_column"]

        for correspondence in structure["candidate_correspondences"]:

            candidate_id = f"C{candidate_counter:03d}"
            candidate_counter += 1

            candidate_evidence_ids = []

            # Source attribute
            eid = f"E{evidence_counter:03d}"
            evidence_counter += 1

            evidence_units.append({
                "id": eid,
                "type": "ATTRIBUTE",
                "model": correspondence["source_model"],
                "column": correspondence["source_column"]
            })

            candidate_evidence_ids.append(eid)

            # Target attribute
            eid = f"E{evidence_counter:03d}"
            evidence_counter += 1

            evidence_units.append({
                "id": eid,
                "type": "ATTRIBUTE",
                "model": correspondence["target_model"],
                "column": correspondence["target_column"]
            })

            candidate_evidence_ids.append(eid)

            # Declared relationship
            eid = f"E{evidence_counter:03d}"
            evidence_counter += 1

            evidence_units.append({
                "id": eid,
                "type": "RELATIONSHIP",
                "source_model": source_model,
                "source_column": relationship_column,
                "target_model": target_model,
                "target_column": target_column
            })

            candidate_evidence_ids.append(eid)

            # Target uniqueness
            eid = f"E{evidence_counter:03d}"
            evidence_counter += 1

            evidence_units.append({
                "id": eid,
                "type": "UNIQUE",
                "model": target_model,
                "column": target_column
            })

            candidate_evidence_ids.append(eid)

            candidates.append({
                "candidate_id": candidate_id,

                "source": {
                    "model": correspondence["source_model"],
                    "column": correspondence["source_column"]
                },

                "target": {
                    "model": correspondence["target_model"],
                    "column": correspondence["target_column"]
                },

                "relationship": {
                    "source_model": source_model,
                    "source_column": relationship_column,
                    "target_model": target_model,
                    "target_column": target_column
                },

                "source_unique_columns":
                    structure["source_unique_columns"],

                "evidence_ids":
                    candidate_evidence_ids
            })

    return {
        "candidates": candidates,
        "evidence_units": evidence_units
    }

In [61]:
# Sanity check — L2 batch candidate payload

l2_candidate_payload = build_l2_candidate_payload(
    l2_structural_candidates
)

print(
    "Candidates:",
    len(l2_candidate_payload["candidates"])
)

print(
    "Evidence units:",
    len(l2_candidate_payload["evidence_units"])
)

print("\n=== CANDIDATES ===")

for candidate in l2_candidate_payload["candidates"]:

    print(
        candidate["candidate_id"],
        f'{candidate["source"]["model"]}.'
        f'{candidate["source"]["column"]}',
        "<->",
        f'{candidate["target"]["model"]}.'
        f'{candidate["target"]["column"]}'
    )

Candidates: 16
Evidence units: 64

=== CANDIDATES ===
C001 stg_synthea__careplans.patient_id <-> stg_synthea__encounters.patient_id
C002 stg_synthea__claims_transactions.department_id <-> stg_synthea__claims.department_id
C003 stg_synthea__claims_transactions.encounter_id <-> stg_synthea__claims.encounter_id
C004 stg_synthea__claims_transactions.patient_id <-> stg_synthea__claims.patient_id
C005 stg_synthea__claims_transactions.provider_id <-> stg_synthea__claims.provider_id
C006 stg_synthea__claims_transactions.supervising_provider_id <-> stg_synthea__claims.supervising_provider_id
C007 stg_synthea__claims_transactions.patient_id <-> stg_synthea__encounters.patient_id
C008 stg_synthea__claims_transactions.provider_id <-> stg_synthea__encounters.provider_id
C009 stg_synthea__claims_transactions.provider_id <-> stg_synthea__providers.provider_id
C010 stg_synthea__claims.provider_id <-> stg_synthea__providers.provider_id
C011 stg_synthea__claims.patient_id <-> stg_synthea__encounters.pat

In [62]:
# Sanity check — neural validation of L2 candidate correspondences

l2_batch_candidate_text = json.dumps(
    l2_candidate_payload,
    indent=2,
    ensure_ascii=False
)

l2_batch_response = invoke_v3_sonnet(
    V3_L2_BATCH_CORRESPONDENCE_PROMPT,
    l2_batch_candidate_text,
    max_tokens=3000
)

l2_batch_result = parse_v3_json_response(
    l2_batch_response
)

print("=== L2 NEURAL CORRESPONDENCES ===\n")

for result in l2_batch_result.get("correspondences", []):
    print(
        result.get("candidate_id"),
        "semantic_correspondence=",
        result.get("semantic_correspondence"),
        "evidence=",
        result.get("evidence_ids"),
        "confidence=",
        result.get("confidence")
    )

=== L2 NEURAL CORRESPONDENCES ===

C001 semantic_correspondence= True evidence= ['E001', 'E002', 'E003', 'E004'] confidence= HIGH
C002 semantic_correspondence= True evidence= ['E005', 'E006', 'E007', 'E008'] confidence= HIGH
C003 semantic_correspondence= True evidence= ['E009', 'E010', 'E011', 'E012'] confidence= HIGH
C004 semantic_correspondence= True evidence= ['E013', 'E014', 'E015', 'E016'] confidence= HIGH
C005 semantic_correspondence= True evidence= ['E017', 'E018', 'E019', 'E020'] confidence= HIGH
C006 semantic_correspondence= True evidence= ['E021', 'E022', 'E023', 'E024'] confidence= HIGH
C007 semantic_correspondence= True evidence= ['E025', 'E026', 'E027', 'E028'] confidence= HIGH
C008 semantic_correspondence= True evidence= ['E029', 'E030', 'E031', 'E032'] confidence= HIGH
C009 semantic_correspondence= True evidence= ['E033', 'E034', 'E035', 'E036'] confidence= HIGH
C010 semantic_correspondence= True evidence= ['E037', 'E038', 'E039', 'E040'] confidence= HIGH
C011 semantic_c

In [63]:
# V3 batch audit — L2_INTRODUCE_TRANSITIVE_DEPENDENCY

def audit_v3_transitive_dependency(audit_input):

    # 1. Deterministically extract relationship structures
    structures = extract_l2_relationship_candidates(
        audit_input
    )

    # 2. Retain structures satisfying the explicit L2 premises
    eligible_structures = filter_l2_structural_candidates(
        structures
    )

    # No structurally viable dependency chain
    if not eligible_structures:
        return {
            "findings": [],
            "candidate_correspondences": [],
            "validated_correspondences": [],
            "eligible_structures": 0
        }

    # 3. Build addressable evidence
    payload = build_l2_candidate_payload(
        eligible_structures
    )

    candidates = payload["candidates"]
    evidence_units = payload["evidence_units"]

    if not candidates:
        return {
            "findings": [],
            "candidate_correspondences": [],
            "validated_correspondences": [],
            "eligible_structures":
                len(eligible_structures)
        }

    # 4. Neural reconstruction of semantic correspondences
    response = invoke_v3_sonnet(
        V3_L2_BATCH_CORRESPONDENCE_PROMPT,
        json.dumps(
            payload,
            ensure_ascii=False,
            indent=2
        ),
        max_tokens=3000
    )

    result = parse_v3_json_response(response)

    neural_results = result.get(
        "correspondences",
        []
    )

    # 5. Deterministically validate candidate and evidence IDs
    candidates_by_id = {
        candidate["candidate_id"]: candidate
        for candidate in candidates
    }

    available_evidence_ids = {
        evidence["id"]
        for evidence in evidence_units
    }

    validated_correspondences = []

    for neural_result in neural_results:

        candidate_id = neural_result.get(
            "candidate_id"
        )

        if candidate_id not in candidates_by_id:
            continue

        if not neural_result.get(
            "semantic_correspondence",
            False
        ):
            continue

        claimed_evidence_ids = set(
            neural_result.get(
                "evidence_ids",
                []
            )
        )

        provenance_valid = (
            len(claimed_evidence_ids) > 0
            and claimed_evidence_ids.issubset(
                available_evidence_ids
            )
        )

        if not provenance_valid:
            continue

        validated_correspondences.append({
            **candidates_by_id[candidate_id],
            "evidence_ids":
                sorted(claimed_evidence_ids),
            "confidence":
                neural_result.get("confidence")
        })

    # 6. Deterministically derive transitive dependencies
    findings = []

    for correspondence in validated_correspondences:

        relationship = correspondence[
            "relationship"
        ]

        for source_key in correspondence[
            "source_unique_columns"
        ]:

            findings.append({
                "finding":
                    "TRANSITIVE_DEPENDENCY",

                "model":
                    relationship["source_model"],

                "source_key":
                    source_key,

                "intermediate_column":
                    relationship["source_column"],

                "dependent_column":
                    correspondence["source"]["column"],

                "target_model":
                    relationship["target_model"],

                "target_key":
                    relationship["target_column"],

                "target_dependent_column":
                    correspondence["target"]["column"],

                "candidate_id":
                    correspondence["candidate_id"],

                "evidence_ids":
                    correspondence["evidence_ids"],

                "confidence":
                    correspondence["confidence"]
            })

    return {
        "findings": findings,
        "candidate_correspondences":
            candidates,
        "validated_correspondences":
            validated_correspondences,
        "eligible_structures":
            len(eligible_structures),
        "raw_response":
            response
    }

In [64]:
# Sanity check — complete L2 neuro-symbolic audit

l2_audit_result = audit_v3_transitive_dependency(
    v3_l2_audit_input
)

print("Eligible structures:",
      l2_audit_result["eligible_structures"])

print("Candidate correspondences:",
      len(l2_audit_result["candidate_correspondences"]))

print("Validated correspondences:",
      len(l2_audit_result["validated_correspondences"]))

print("Findings:",
      len(l2_audit_result["findings"]))

print("\n=== TARGET FINDINGS ===\n")

target_findings = [
    finding
    for finding in l2_audit_result["findings"]
    if (
        finding["model"]
        == "stg_synthea__providers"

        and finding["source_key"]
        == "provider_id"

        and finding["intermediate_column"]
        == "organization_id"

        and finding["dependent_column"]
        == "organization_state"

        and finding["target_model"]
        == "stg_synthea__organizations"

        and finding["target_key"]
        == "organization_id"

        and finding["target_dependent_column"]
        == "organization_state"
    )
]

for finding in target_findings:
    print(json.dumps(
        finding,
        indent=2,
        ensure_ascii=False
    ))

print(
    "\nTarget transitive dependency detected:",
    len(target_findings) > 0
)

Eligible structures: 10
Candidate correspondences: 16
Validated correspondences: 16
Findings: 16

=== TARGET FINDINGS ===

{
  "finding": "TRANSITIVE_DEPENDENCY",
  "model": "stg_synthea__providers",
  "source_key": "provider_id",
  "intermediate_column": "organization_id",
  "dependent_column": "organization_state",
  "target_model": "stg_synthea__organizations",
  "target_key": "organization_id",
  "target_dependent_column": "organization_state",
  "candidate_id": "C016",
  "evidence_ids": [
    "E061",
    "E062",
    "E063",
    "E064"
  ],
  "confidence": "HIGH"
}

Target transitive dependency detected: True


In [65]:
# Prepare V3 batch — L2_INTRODUCE_TRANSITIVE_DEPENDENCY

L2_BATCH_DIR = Path(
    "rdbscore_generated_v2/L2/"
    "L2_INTRODUCE_TRANSITIVE_DEPENDENCY"
)

L2_V3_RESULTS_PATH = Path(
    "v3_l2_transitive_dependency_results.jsonl"
)

l2_case_dirs = sorted([
    path
    for path in L2_BATCH_DIR.iterdir()
    if path.is_dir()
])

print("L2 benchmark instances:", len(l2_case_dirs))
print("Results file:", L2_V3_RESULTS_PATH)

L2 benchmark instances: 349
Results file: v3_l2_transitive_dependency_results.jsonl


In [66]:
# Run V3 batch — L2_INTRODUCE_TRANSITIVE_DEPENDENCY

completed_cases = set()

if L2_V3_RESULTS_PATH.exists():
    with L2_V3_RESULTS_PATH.open(
        "r",
        encoding="utf-8"
    ) as f:
        for line in f:
            if not line.strip():
                continue

            record = json.loads(line)

            if record.get("status") == "OK":
                completed_cases.add(
                    record["case_id"]
                )

print(
    "Already completed:",
    len(completed_cases)
)

remaining_cases = [
    case_dir
    for case_dir in l2_case_dirs
    if case_dir.name not in completed_cases
]

print(
    "Remaining:",
    len(remaining_cases)
)

for index, case_dir in enumerate(
    remaining_cases,
    start=1
):

    case_id = case_dir.name

    try:
        audit_input_path = (
            case_dir / "audit_input.txt"
        )

        audit_input = audit_input_path.read_text(
            encoding="utf-8"
        )

        result = audit_v3_transitive_dependency(
            audit_input
        )

        record = {
            "case_id": case_id,
            "status": "OK",
            "eligible_structures":
                result["eligible_structures"],
            "candidate_correspondences":
                result["candidate_correspondences"],
            "validated_correspondences":
                result["validated_correspondences"],
            "findings":
                result["findings"]
        }

    except Exception as e:

        record = {
            "case_id": case_id,
            "status": "ERROR",
            "error": str(e)
        }

    with L2_V3_RESULTS_PATH.open(
        "a",
        encoding="utf-8"
    ) as f:
        f.write(
            json.dumps(
                record,
                ensure_ascii=False
            )
            + "\n"
        )

    print(
        f"[{index}/{len(remaining_cases)}]",
        case_id,
        record["status"],
        "findings=",
        len(record.get("findings", []))
    )

Already completed: 349
Remaining: 0


In [67]:
# Evaluate V3 — L2_INTRODUCE_TRANSITIVE_DEPENDENCY against ground truth

latest_ok_by_case = {}

with L2_V3_RESULTS_PATH.open(
    "r",
    encoding="utf-8"
) as f:
    for line in f:
        if not line.strip():
            continue

        record = json.loads(line)

        if record.get("status") == "OK":
            latest_ok_by_case[
                record["case_id"]
            ] = record


evaluation_rows = []

for case_id, record in latest_ok_by_case.items():

    case_dir = L2_BATCH_DIR / case_id

    ground_truth_path = (
        case_dir / "ground_truth.json"
    )

    ground_truth = json.loads(
        ground_truth_path.read_text(
            encoding="utf-8"
        )
    )

    findings = record.get(
        "findings",
        []
    )

    candidate = ground_truth["candidate"]

    print("\nCASE:", case_id)
    print("GROUND TRUTH CANDIDATE:")
    print(json.dumps(
        candidate,
        indent=2,
        ensure_ascii=False
    ))

    print("\nFIRST FINDING:")
    print(json.dumps(
        findings[0] if findings else None,
        indent=2,
        ensure_ascii=False
    ))

    break


CASE: L2_INTRODUCE_TRANSITIVE_DEPENDENCY_013b469473fc
GROUND TRUTH CANDIDATE:
{
  "source_model": "stg_synthea__providers",
  "source_key": "provider_id",
  "source_fk": "organization_id",
  "target_model": "stg_synthea__organizations",
  "target_key": "organization_id",
  "target_column": "organization_state",
  "evidence": {
    "type": "TRANSITIVE_DEPENDENCY_EVIDENCE",
    "source_key_unique": true,
    "source_fk_unique": false,
    "relationship": {
      "column": "organization_id",
      "to": "ref('stg_synthea__organizations')",
      "field": "organization_id"
    },
    "target_key_unique": true,
    "dependency_chain": [
      {
        "determinant": "provider_id",
        "dependent": "organization_id",
        "basis": "SOURCE_KEY_UNIQUENESS"
      },
      {
        "determinant": "organization_id",
        "dependent": "organization_state",
        "basis": "RELATIONSHIP_TO_UNIQUE_TARGET_KEY"
      }
    ]
  }
}

FIRST FINDING:
{
  "finding": "TRANSITIVE_DEPENDENCY",
 

In [68]:
# Evaluate V3 — L2_INTRODUCE_TRANSITIVE_DEPENDENCY against ground truth

latest_ok_by_case = {}

with L2_V3_RESULTS_PATH.open(
    "r",
    encoding="utf-8"
) as f:
    for line in f:
        if not line.strip():
            continue

        record = json.loads(line)

        if record.get("status") == "OK":
            latest_ok_by_case[
                record["case_id"]
            ] = record


evaluation_rows = []

for case_id, record in latest_ok_by_case.items():

    case_dir = L2_BATCH_DIR / case_id

    ground_truth = json.loads(
        (case_dir / "ground_truth.json").read_text(
            encoding="utf-8"
        )
    )

    gt = ground_truth["candidate"]

    findings = record.get(
        "findings",
        []
    )

    matching_findings = [
        finding
        for finding in findings
        if (
            finding.get("finding")
            == "TRANSITIVE_DEPENDENCY"

            and finding.get("model")
            == gt["source_model"]

            and finding.get("source_key")
            == gt["source_key"]

            and finding.get("intermediate_column")
            == gt["source_fk"]

            and finding.get("dependent_column")
            == gt["target_column"]

            and finding.get("target_model")
            == gt["target_model"]

            and finding.get("target_key")
            == gt["target_key"]

            and finding.get("target_dependent_column")
            == gt["target_column"]
        )
    ]

    target_detected = (
        len(matching_findings) > 0
    )

    exact_case_match = (
        target_detected
        and len(findings) == 1
    )

    evaluation_rows.append({
        "case_id": case_id,
        "findings": len(findings),
        "target_detected":
            target_detected,
        "exact_case_match":
            exact_case_match
    })


n = len(evaluation_rows)

target_detected_count = sum(
    row["target_detected"]
    for row in evaluation_rows
)

exact_case_match_count = sum(
    row["exact_case_match"]
    for row in evaluation_rows
)

target_not_detected_count = (
    n - target_detected_count
)


print("=== V3 L2 EVALUATION ===\n")

print("N:", n)

print(
    "Target detected:",
    target_detected_count,
    f"({target_detected_count / n:.2%})"
)

print(
    "Exact case match:",
    exact_case_match_count,
    f"({exact_case_match_count / n:.2%})"
)

print(
    "Target not detected:",
    target_not_detected_count,
    f"({target_not_detected_count / n:.2%})"
)

=== V3 L2 EVALUATION ===

N: 349
Target detected: 349 (100.00%)
Exact case match: 2 (0.57%)
Target not detected: 0 (0.00%)


### 8.3 L3 — Semantic Relationship Mismatch

We now evaluate the neuro-symbolic auditing procedure for L3 semantic
schema defects using `L3_INTRODUCE_SEMANTIC_RELATIONSHIP_MISMATCH`.

For each benchmark instance, the procedure separates the declared
relationship from the evidence used for semantic reconstruction. The neural
component reconstructs the expected relationship from the remaining schema
evidence, while deterministic verification compares the reconstructed
expectation with the relationship actually declared in the mutated schema.

The auditing procedure follows:

$$
E^{-declared}
\xrightarrow{\text{Neural Reconstruction}}
ExpectedReference(A,B)
$$

$$
Schema
\xrightarrow{\text{Deterministic Extraction}}
DeclaredReference(A,C)
$$

$$
ExpectedReference(A,B)
\land
DeclaredReference(A,C)
\land
B \neq C
\rightarrow
SemanticRelationshipMismatch(A,B,C)
$$

As in the previous experiments, mutation metadata and ground truth are not
included in the model-visible evidence and are used only during evaluation.

In [69]:
# L3 batch support — extract declared relationship structures

def extract_declared_relationships(audit_input):

    metadata = extract_schema_metadata(
        audit_input
    )

    relationships = []

    for model in metadata.get("models", []):

        model_name = model.get("name")

        for column in model.get("columns", []):

            column_name = column.get("name")

            for test in column.get("tests", []) or []:

                if not isinstance(test, dict):
                    continue

                relationship = test.get(
                    "relationships"
                )

                if not relationship:
                    continue

                relationships.append({
                    "source_model":
                        model_name,

                    "source_column":
                        column_name,

                    "declared_target":
                        relationship.get("to"),

                    "declared_target_column":
                        relationship.get("field")
                })

    return relationships

In [70]:
# Sanity check — declared relationships in exploratory L3 case

l3_declared_relationships = (
    extract_declared_relationships(
        v3_l3_audit_input
    )
)

print(
    "Declared relationships:",
    len(l3_declared_relationships)
)

for relationship in l3_declared_relationships:

    print(
        relationship["source_model"],
        ".",
        relationship["source_column"],
        " -> ",
        relationship["declared_target"],
        ".",
        relationship["declared_target_column"],
        sep=""
    )

Declared relationships: 4
dim_listings_hosts.HOST_ID -> {{ ref('wow_agg_reviews') }}.date_sentiment_id
dim_listings_hosts.LISTING_ID -> ref('dim_listings').LISTING_ID
fct_reviews.LISTING_ID -> ref('dim_listings').LISTING_ID
fct_reviews.LISTING_ID -> ref('dim_listings_hosts').LISTING_ID


In [71]:
# L3 batch support — remove one declared relationship from model-visible evidence

import copy


def remove_declared_relationship(
    audit_input,
    source_model,
    source_column,
    declared_target,
    declared_target_column
):

    metadata = extract_schema_metadata(
        audit_input
    )

    evidence_metadata = copy.deepcopy(
        metadata
    )

    removed = False

    for model in evidence_metadata.get(
        "models", []
    ):

        if model.get("name") != source_model:
            continue

        for column in model.get(
            "columns", []
        ):

            if column.get("name") != source_column:
                continue

            retained_tests = []

            for test in column.get(
                "tests", []
            ) or []:

                remove_test = False

                if isinstance(test, dict):

                    relationship = test.get(
                        "relationships"
                    )

                    if relationship:

                        if (
                            relationship.get("to")
                            == declared_target
                            and relationship.get("field")
                            == declared_target_column
                        ):
                            remove_test = True
                            removed = True

                if not remove_test:
                    retained_tests.append(test)

            column["tests"] = retained_tests

    return evidence_metadata, removed

In [72]:
# Sanity check — remove mutated L3 relationship from neural evidence

host_relationship = next(
    relationship
    for relationship in l3_declared_relationships
    if (
        relationship["source_model"]
        == "dim_listings_hosts"
        and relationship["source_column"]
        == "HOST_ID"
    )
)

l3_evidence_without_declared, removed = (
    remove_declared_relationship(
        v3_l3_audit_input,
        host_relationship["source_model"],
        host_relationship["source_column"],
        host_relationship["declared_target"],
        host_relationship["declared_target_column"]
    )
)

print("Relationship removed:", removed)

print("\n=== HOST_ID AFTER REMOVAL ===\n")

for model in l3_evidence_without_declared.get(
    "models", []
):
    if model.get("name") != "dim_listings_hosts":
        continue

    for column in model.get("columns", []):
        if column.get("name") == "HOST_ID":
            print(
                json.dumps(
                    column,
                    indent=2,
                    ensure_ascii=False
                )
            )

Relationship removed: True

=== HOST_ID AFTER REMOVAL ===

{
  "name": "HOST_ID",
  "tests": [
    "not_null"
  ]
}


In [73]:
# L3 batch support — reconstruct expected reference

def reconstruct_expected_reference_l3(
    audit_input,
    declared_relationship
):

    # 1. Remove the relationship being evaluated
    evidence_metadata, removed = (
        remove_declared_relationship(
            audit_input,
            declared_relationship["source_model"],
            declared_relationship["source_column"],
            declared_relationship["declared_target"],
            declared_relationship["declared_target_column"]
        )
    )

    if not removed:
        raise ValueError(
            "Declared relationship could not be removed"
        )

    # 2. Serialize the remaining schema evidence
    reconstruction_evidence = yaml.safe_dump(
        evidence_metadata,
        sort_keys=False,
        allow_unicode=True
    )

    # 3. Build the same model-visible input used
    #    in the exploratory L3 experiment
    expected_reference_input = f"""
SCHEMA METADATA:

{reconstruction_evidence}

SOURCE ATTRIBUTE:

model: {declared_relationship["source_model"]}
column: {declared_relationship["source_column"]}
"""

    # 4. Neural reconstruction
    response = invoke_v3_sonnet(
        V3_EXPECTED_REFERENCE_PROMPT,
        expected_reference_input
    )

    result = parse_v3_json_response(
        response
    )

    return {
        "expected_reference": result,
        "raw_response": response
    }

In [74]:
# Sanity check — generalized L3 expected-reference reconstruction

l3_reconstruction_result = (
    reconstruct_expected_reference_l3(
        v3_l3_audit_input,
        host_relationship
    )
)

print("=== RECONSTRUCTED EXPECTED REFERENCE ===\n")

print(
    json.dumps(
        l3_reconstruction_result[
            "expected_reference"
        ],
        indent=2,
        ensure_ascii=False
    )
)

=== RECONSTRUCTED EXPECTED REFERENCE ===

{
  "source_model": "dim_listings_hosts",
  "source_column": "HOST_ID",
  "expected_target_model": "dim_hosts",
  "expected_target_column": "HOST_ID",
  "evidence": "The dim_listings_hosts model contains a HOST_ID column, and there is a separate dim_hosts model with a HOST_ID column that is marked as unique and not null. This suggests that the HOST_ID in dim_listings_hosts is a foreign key reference to the HOST_ID primary key in the dim_hosts model.",
  "confidence": "HIGH"
}


In [75]:
# L3 batch support — normalize declared target model

def normalize_relationship_target(target):

    if target is None:
        return None

    return (
        target
        .replace("{{", "")
        .replace("}}", "")
        .replace("ref(", "")
        .replace(")", "")
        .replace("'", "")
        .replace('"', "")
        .strip()
    )

In [76]:
# V3 batch audit — L3_INTRODUCE_SEMANTIC_RELATIONSHIP_MISMATCH

def audit_v3_semantic_relationship_mismatch(
    audit_input
):

    # 1. Deterministically extract declared relationships
    declared_relationships = (
        extract_declared_relationships(
            audit_input
        )
    )

    findings = []
    reconstructions = []

    # 2. Evaluate each declared relationship independently
    for declared in declared_relationships:

        reconstruction = (
            reconstruct_expected_reference_l3(
                audit_input,
                declared
            )
        )

        expected = reconstruction[
            "expected_reference"
        ]

        expected_target_model = expected.get(
            "expected_target_model"
        )

        expected_target_column = expected.get(
            "expected_target_column"
        )

        declared_target_model = (
            normalize_relationship_target(
                declared["declared_target"]
            )
        )

        declared_target_column = declared[
            "declared_target_column"
        ]

        reconstructions.append({
            "source_model":
                declared["source_model"],
            "source_column":
                declared["source_column"],
            "expected_target_model":
                expected_target_model,
            "expected_target_column":
                expected_target_column,
            "declared_target_model":
                declared_target_model,
            "declared_target_column":
                declared_target_column,
            "confidence":
                expected.get("confidence"),
            "evidence":
                expected.get("evidence")
        })

        # Insufficient evidence to establish
        # an expected relationship
        if (
            expected_target_model is None
            or expected_target_column is None
        ):
            continue

        # 3. Deterministic verification
        mismatch = (
            expected_target_model.lower()
            != declared_target_model.lower()
            or
            expected_target_column.lower()
            != declared_target_column.lower()
        )

        # 4. Produce finding only when expectation
        #    and declaration disagree
        if mismatch:

            findings.append({
                "finding":
                    "SEMANTIC_RELATIONSHIP_MISMATCH",

                "source_model":
                    declared["source_model"],

                "source_column":
                    declared["source_column"],

                "expected_target_model":
                    expected_target_model,

                "expected_target_column":
                    expected_target_column,

                "declared_target_model":
                    declared_target_model,

                "declared_target_column":
                    declared_target_column,

                "evidence":
                    expected.get("evidence"),

                "confidence":
                    expected.get("confidence")
            })

    return {
        "findings": findings,
        "reconstructions": reconstructions,
        "declared_relationships":
            declared_relationships
    }

In [77]:
# Sanity check — complete L3 neuro-symbolic audit

l3_audit_result = (
    audit_v3_semantic_relationship_mismatch(
        v3_l3_audit_input
    )
)

print(
    "Declared relationships:",
    len(l3_audit_result["declared_relationships"])
)

print(
    "Reconstructions:",
    len(l3_audit_result["reconstructions"])
)

print(
    "Findings:",
    len(l3_audit_result["findings"])
)


target_findings = [
    finding
    for finding in l3_audit_result["findings"]
    if (
        finding["source_model"]
        == "dim_listings_hosts"

        and finding["source_column"]
        == "HOST_ID"

        and finding["expected_target_model"]
        == "dim_hosts"

        and finding["expected_target_column"]
        == "HOST_ID"

        and finding["declared_target_model"]
        == "wow_agg_reviews"

        and finding["declared_target_column"]
        == "date_sentiment_id"
    )
]


print("\n=== TARGET FINDINGS ===\n")

for finding in target_findings:
    print(
        json.dumps(
            finding,
            indent=2,
            ensure_ascii=False
        )
    )


print(
    "\nTarget semantic relationship mismatch detected:",
    len(target_findings) > 0
)

Declared relationships: 4
Reconstructions: 4
Findings: 3

=== TARGET FINDINGS ===

{
  "finding": "SEMANTIC_RELATIONSHIP_MISMATCH",
  "source_model": "dim_listings_hosts",
  "source_column": "HOST_ID",
  "expected_target_model": "dim_hosts",
  "expected_target_column": "HOST_ID",
  "declared_target_model": "wow_agg_reviews",
  "declared_target_column": "date_sentiment_id",
  "evidence": "The dim_listings_hosts model contains a HOST_ID column, and there is a separate dim_hosts model with a HOST_ID column that is marked as unique and not null. This suggests that the HOST_ID in dim_listings_hosts is a foreign key reference to the HOST_ID primary key in the dim_hosts model.",
  "confidence": "HIGH"
}

Target semantic relationship mismatch detected: True


In [78]:
# Inspect L3 benchmark directory structure

L3_ROOT_DIR = Path(
    "rdbscore_generated_v2/L3"
)

print("Exists:", L3_ROOT_DIR.exists())

print("\n=== L3 DIRECTORY CONTENTS ===\n")

for path in sorted(L3_ROOT_DIR.iterdir()):
    print(
        "[DIR]" if path.is_dir() else "[FILE]",
        path.name
    )

Exists: True

=== L3 DIRECTORY CONTENTS ===

[DIR] L3_REL_0001_airbnb001_dim_listings_hosts_host_id
[DIR] L3_REL_0002_airbnb001_dim_listings_hosts_listing_id
[DIR] L3_REL_0003_airbnb001_fct_reviews_listing_id
[DIR] L3_REL_0004_airbnb001_fct_reviews_listing_id
[DIR] L3_REL_0005_airbnb002_dim_listings_hosts_host_id
[DIR] L3_REL_0006_airbnb002_dim_listings_hosts_listing_id
[DIR] L3_REL_0007_airbnb002_fct_reviews_listing_id
[DIR] L3_REL_0008_airbnb002_fct_reviews_listing_id
[DIR] L3_REL_0009_synthea001_stg_synthea_allergies_patient_id
[DIR] L3_REL_0010_synthea001_stg_synthea_allergies_encounter_id
[DIR] L3_REL_0011_synthea001_stg_synthea_careplans_patient_id
[DIR] L3_REL_0012_synthea001_stg_synthea_careplans_encounter_id
[DIR] L3_REL_0013_synthea001_stg_synthea_claims_transactions_claim_id
[DIR] L3_REL_0014_synthea001_stg_synthea_claims_transactions_patient_id
[DIR] L3_REL_0015_synthea001_stg_synthea_claims_transactions_place_of_service
[DIR] L3_REL_0016_synthea001_stg_synthea_claims_trans

In [79]:
# L3 batch setup

L3_BATCH_DIR = Path(
    "rdbscore_generated_v2/L3"
)

L3_V3_RESULTS_PATH = Path(
    "v3_l3_semantic_relationship_mismatch_results.jsonl"
)

l3_case_dirs = sorted([
    path
    for path in L3_BATCH_DIR.iterdir()
    if (
        path.is_dir()
        and path.name.startswith("L3_REL_")
    )
])

print(
    "L3 benchmark instances:",
    len(l3_case_dirs)
)

print(
    "First case:",
    l3_case_dirs[0].name
)

print(
    "Last case:",
    l3_case_dirs[-1].name
)

print(
    "Results file:",
    L3_V3_RESULTS_PATH
)

L3 benchmark instances: 51
First case: L3_REL_0001_airbnb001_dim_listings_hosts_host_id
Last case: L3_REL_0051_synthea001_stg_synthea_supplies_encounter_id
Results file: v3_l3_semantic_relationship_mismatch_results.jsonl


In [80]:
# L3 batch runner — incremental execution with resume

def load_completed_l3_cases(results_path):

    completed = set()

    if not results_path.exists():
        return completed

    with results_path.open(
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:

            if not line.strip():
                continue

            record = json.loads(line)

            if record.get("status") == "OK":
                completed.add(
                    record["case_id"]
                )

    return completed

In [81]:
# Run V3 L3 batch evaluation

completed_l3_cases = load_completed_l3_cases(
    L3_V3_RESULTS_PATH
)

remaining_l3_cases = [
    case_dir
    for case_dir in l3_case_dirs
    if case_dir.name not in completed_l3_cases
]

print(
    "Already completed:",
    len(completed_l3_cases)
)

print(
    "Remaining:",
    len(remaining_l3_cases)
)


for index, case_dir in enumerate(
    remaining_l3_cases,
    start=1
):

    case_id = case_dir.name

    print(
        f"[{index}/{len(remaining_l3_cases)}] "
        f"{case_id}"
    )

    try:

        audit_input_path = (
            case_dir / "audit_input.txt"
        )

        audit_input = audit_input_path.read_text(
            encoding="utf-8"
        )

        result = (
            audit_v3_semantic_relationship_mismatch(
                audit_input
            )
        )

        record = {
            "case_id": case_id,
            "status": "OK",
            "findings": result["findings"],
            "reconstructions":
                result["reconstructions"],
            "declared_relationships":
                result["declared_relationships"]
        }

        print(
            "  OK —",
            len(result["findings"]),
            "finding(s)"
        )

    except Exception as e:

        record = {
            "case_id": case_id,
            "status": "ERROR",
            "error_type":
                type(e).__name__,
            "error":
                str(e)
        }

        print(
            "  ERROR —",
            type(e).__name__,
            str(e)
        )

    with L3_V3_RESULTS_PATH.open(
        "a",
        encoding="utf-8"
    ) as f:

        f.write(
            json.dumps(
                record,
                ensure_ascii=False
            )
            + "\n"
        )

Already completed: 51
Remaining: 0


In [82]:
# Consolidate L3 batch results — keep the latest successful result per case

latest_l3_ok_by_case = {}
l3_errors = []

with L3_V3_RESULTS_PATH.open(
    "r",
    encoding="utf-8"
) as f:

    for line in f:

        if not line.strip():
            continue

        record = json.loads(line)

        if record.get("status") == "OK":
            latest_l3_ok_by_case[
                record["case_id"]
            ] = record

        elif record.get("status") == "ERROR":
            l3_errors.append(record)


print(
    "Successful unique cases:",
    len(latest_l3_ok_by_case)
)

print(
    "Expected cases:",
    len(l3_case_dirs)
)

print(
    "Error records in JSONL:",
    len(l3_errors)
)


missing_l3_cases = [
    case_dir.name
    for case_dir in l3_case_dirs
    if case_dir.name not in latest_l3_ok_by_case
]

print(
    "Cases without successful result:",
    len(missing_l3_cases)
)

if missing_l3_cases:
    print("\nMissing:")
    for case_id in missing_l3_cases:
        print(case_id)

Successful unique cases: 51
Expected cases: 51
Error records in JSONL: 16
Cases without successful result: 0


In [83]:
# Inspect L3 ground-truth manifest structure

L3_MANIFEST_PATH = (
    L3_BATCH_DIR / "generation_manifest_l3.json"
)

with L3_MANIFEST_PATH.open(
    "r",
    encoding="utf-8"
) as f:
    l3_manifest = json.load(f)


print(
    "Manifest type:",
    type(l3_manifest).__name__
)

if isinstance(l3_manifest, dict):

    print(
        "Top-level keys:",
        list(l3_manifest.keys())
    )

    for key, value in l3_manifest.items():

        print(
            "\nKEY:",
            key,
            "| TYPE:",
            type(value).__name__
        )

        if isinstance(value, list) and value:
            print(
                json.dumps(
                    value[0],
                    indent=2,
                    ensure_ascii=False
                )
            )
            break

elif isinstance(l3_manifest, list):

    print(
        "Manifest entries:",
        len(l3_manifest)
    )

    if l3_manifest:
        print(
            json.dumps(
                l3_manifest[0],
                indent=2,
                ensure_ascii=False
            )
        )

Manifest type: list
Manifest entries: 51
{
  "instance_id": "L3_REL_0001_airbnb001_dim_listings_hosts_host_id",
  "level": "L3",
  "operator_id": "L3_INTRODUCE_SEMANTIC_RELATIONSHIP_MISMATCH",
  "project": "airbnb001",
  "source_model": "dim_listings_hosts",
  "source_column": "host_id",
  "original_target": "dim_hosts",
  "mutated_target": "wow_agg_reviews",
  "mutated_target_key": "date_sentiment_id",
  "instance_dir": "rdbscore_generated_v2/L3/L3_REL_0001_airbnb001_dim_listings_hosts_host_id",
  "audit_input_path": "rdbscore_generated_v2/L3/L3_REL_0001_airbnb001_dim_listings_hosts_host_id/audit_input.txt",
  "ground_truth_path": "rdbscore_generated_v2/L3/L3_REL_0001_airbnb001_dim_listings_hosts_host_id/ground_truth.json",
  "mutated_schema_path": "rdbscore_generated_v2/L3/L3_REL_0001_airbnb001_dim_listings_hosts_host_id/mutated_schema.yml",
  "prompt_version": "RDBSCORE_L3_PROMPT_V2"
}


In [84]:
# Inspect one L3 ground-truth instance

first_l3_manifest_entry = l3_manifest[0]

first_l3_gt_path = Path(
    first_l3_manifest_entry[
        "ground_truth_path"
    ]
)

with first_l3_gt_path.open(
    "r",
    encoding="utf-8"
) as f:
    first_l3_ground_truth = json.load(f)


print(
    json.dumps(
        first_l3_ground_truth,
        indent=2,
        ensure_ascii=False
    )
)

{
  "protocol_version": "RDBSCORE_V2",
  "level": "L3",
  "operator_id": "L3_INTRODUCE_SEMANTIC_RELATIONSHIP_MISMATCH",
  "defect_family": "Semantic Schema Defects",
  "defect_type": "Semantic Relationship Mismatch",
  "project": "airbnb001",
  "source_model": "dim_listings_hosts",
  "source_column": "host_id",
  "semantic_expectation": {
    "intended_target_entity": "dim_hosts",
    "intended_target_key": "host_id"
  },
  "evidence": {
    "original_relationship": "dim_listings_hosts.host_id -> dim_hosts.host_id",
    "evidence_type": "explicit original semantic correspondence"
  },
  "mutation": {
    "original_target": "dim_hosts",
    "original_target_key": "host_id",
    "mutated_target": "wow_agg_reviews",
    "mutated_target_key": "date_sentiment_id",
    "mutated_target_key_not_null": true,
    "target_semantic_similarity": 0.0,
    "alternative_count": 5,
    "selection_index": 4,
    "selection_method": "SHA256 deterministic diversified selection"
  },
  "expected_violation"

In [85]:
# L3 evaluation support — match finding against ground truth

def normalize_l3_value(value):

    if value is None:
        return None

    return str(value).strip().lower()


def normalize_l3_model(value):

    value = normalize_l3_value(value)

    if value is None:
        return None

    # The L3 ground truth uses a single underscore in
    # model prefixes where the dbt schema uses "__".
    return value.replace("__", "_")


def l3_finding_matches_ground_truth(
    finding,
    ground_truth
):

    semantic_expectation = ground_truth[
        "semantic_expectation"
    ]

    mutation = ground_truth[
        "mutation"
    ]

    return (
        normalize_l3_value(
            finding.get("finding")
        )
        ==
        normalize_l3_value(
            ground_truth[
                "expected_violation"
            ]["type"]
        )

        and normalize_l3_model(
            finding.get("source_model")
        )
        ==
        normalize_l3_model(
            ground_truth["source_model"]
        )

        and normalize_l3_value(
            finding.get("source_column")
        )
        ==
        normalize_l3_value(
            ground_truth["source_column"]
        )

        and normalize_l3_model(
            finding.get(
                "expected_target_model"
            )
        )
        ==
        normalize_l3_model(
            semantic_expectation[
                "intended_target_entity"
            ]
        )

        and normalize_l3_value(
            finding.get(
                "expected_target_column"
            )
        )
        ==
        normalize_l3_value(
            semantic_expectation[
                "intended_target_key"
            ]
        )

        and normalize_l3_model(
            finding.get(
                "declared_target_model"
            )
        )
        ==
        normalize_l3_model(
            mutation["mutated_target"]
        )

        and normalize_l3_value(
            finding.get(
                "declared_target_column"
            )
        )
        ==
        normalize_l3_value(
            mutation["mutated_target_key"]
        )
    )

In [86]:
# Evaluate V3 L3 target detection and exact case match

l3_evaluation = []

for manifest_entry in l3_manifest:

    case_id = manifest_entry[
        "instance_id"
    ]

    result_record = latest_l3_ok_by_case[
        case_id
    ]

    ground_truth_path = Path(
        manifest_entry[
            "ground_truth_path"
        ]
    )

    with ground_truth_path.open(
        "r",
        encoding="utf-8"
    ) as f:
        ground_truth = json.load(f)

    findings = result_record.get(
        "findings",
        []
    )

    matching_findings = [
        finding
        for finding in findings
        if l3_finding_matches_ground_truth(
            finding,
            ground_truth
        )
    ]

    target_detected = (
        len(matching_findings) > 0
    )

    exact_case_match = (
        target_detected
        and len(findings) == 1
    )

    l3_evaluation.append({
        "case_id": case_id,
        "target_detected":
            target_detected,
        "exact_case_match":
            exact_case_match,
        "finding_count":
            len(findings),
        "matching_finding_count":
            len(matching_findings)
    })


l3_n = len(l3_evaluation)

l3_target_detected = sum(
    row["target_detected"]
    for row in l3_evaluation
)

l3_exact_case_match = sum(
    row["exact_case_match"]
    for row in l3_evaluation
)

l3_target_not_detected = (
    l3_n - l3_target_detected
)


print(
    "=== V3 L3 EVALUATION ===\n"
)

print(
    "N:",
    l3_n
)

print(
    "Target detected:",
    l3_target_detected,
    f"({100 * l3_target_detected / l3_n:.2f}%)"
)

print(
    "Exact case match:",
    l3_exact_case_match,
    f"({100 * l3_exact_case_match / l3_n:.2f}%)"
)

print(
    "Target not detected:",
    l3_target_not_detected,
    f"({100 * l3_target_not_detected / l3_n:.2f}%)"
)

=== V3 L3 EVALUATION ===

N: 51
Target detected: 47 (92.16%)
Exact case match: 0 (0.00%)
Target not detected: 4 (7.84%)


In [87]:
# Inspect L3 cases with no source reconstruction

#no_source_cases = [
#    row
#    for row in l3_diagnostics
#    if row["status"]
#    == "NO_SOURCE_RECONSTRUCTION"
#]

#print(
#    "NO_SOURCE_RECONSTRUCTION cases:",
#    len(no_source_cases)
#)

#print(
#    "\n=== FIRST 5 CASES ===\n"
#)

#for row in no_source_cases[:5]:

#    case_id = row["case_id"]

#    manifest_entry = next(
#        entry
#        for entry in l3_manifest
#        if entry["instance_id"] == case_id
#    )

#    result_record = (
#        latest_l3_ok_by_case[
#            case_id
#        ]
#    )

#    print("=" * 80)
#    print("CASE:", case_id)

#    print(
#        "GT SOURCE:",
#        manifest_entry["source_model"],
#        ".",
#        manifest_entry["source_column"],
#        sep=""
#    )

#    print("\nDECLARED RELATIONSHIPS:")

#    for relationship in result_record.get(
#        "declared_relationships",
#        []
#    ):
#        print(
#            " ",
#            relationship.get("source_model"),
#            ".",
#            relationship.get("source_column"),
#            " -> ",
#            relationship.get(
#                "declared_target"
#            ),
#            ".",
#            relationship.get(
#                "declared_target_column"
#            ),
#            sep=""
#        )

#    print("\nRECONSTRUCTIONS:")

#    for reconstruction in result_record.get(
#        "reconstructions",
#        []
#    ):
#        print(
#            " ",
#            reconstruction.get(
#                "source_model"
#            ),
#            ".",
#            reconstruction.get(
#                "source_column"
#            ),
#            " -> ",
#            reconstruction.get(
#                "expected_target_model"
#            ),
#            ".",
#            reconstruction.get(
#                "expected_target_column"
#            ),
#            sep=""
#        )

#    print()

In [88]:
# Inspect the four L3 target misses

l3_missed_cases = [
    row
    for row in l3_evaluation
    if not row["target_detected"]
]

print(
    "L3 target misses:",
    len(l3_missed_cases)
)

for row in l3_missed_cases:

    case_id = row["case_id"]

    manifest_entry = next(
        entry
        for entry in l3_manifest
        if entry["instance_id"] == case_id
    )

    result_record = (
        latest_l3_ok_by_case[
            case_id
        ]
    )

    ground_truth_path = Path(
        manifest_entry[
            "ground_truth_path"
        ]
    )

    with ground_truth_path.open(
        "r",
        encoding="utf-8"
    ) as f:
        ground_truth = json.load(f)

    gt_source_model = normalize_l3_model(
        ground_truth["source_model"]
    )

    gt_source_column = normalize_l3_value(
        ground_truth["source_column"]
    )

    source_reconstructions = [
        reconstruction
        for reconstruction
        in result_record.get(
            "reconstructions",
            []
        )
        if (
            normalize_l3_model(
                reconstruction.get(
                    "source_model"
                )
            )
            == gt_source_model

            and normalize_l3_value(
                reconstruction.get(
                    "source_column"
                )
            )
            == gt_source_column
        )
    ]

    print("=" * 80)
    print("CASE:", case_id)

    print(
        "EXPECTED:",
        ground_truth[
            "semantic_expectation"
        ][
            "intended_target_entity"
        ],
        ".",
        ground_truth[
            "semantic_expectation"
        ][
            "intended_target_key"
        ],
        sep=""
    )

    print(
        "MUTATED:",
        ground_truth[
            "mutation"
        ][
            "mutated_target"
        ],
        ".",
        ground_truth[
            "mutation"
        ][
            "mutated_target_key"
        ],
        sep=""
    )

    print(
        "SOURCE RECONSTRUCTION:"
    )

    if not source_reconstructions:
        print("  NONE")
    else:
        for reconstruction in source_reconstructions:
            print(
                json.dumps(
                    reconstruction,
                    indent=2,
                    ensure_ascii=False
                )
            )

    print()

L3 target misses: 4
CASE: L3_REL_0003_airbnb001_fct_reviews_listing_id
EXPECTED:dim_listings.listing_id
MUTATED:mom_agg_reviews.date_sentiment_id
SOURCE RECONSTRUCTION:
{
  "source_model": "fct_reviews",
  "source_column": "LISTING_ID",
  "expected_target_model": "dim_listings_hosts",
  "expected_target_column": "LISTING_ID",
  "declared_target_model": "mom_agg_reviews",
  "declared_target_column": "date_sentiment_id",
  "confidence": "HIGH",
  "evidence": "The fct_reviews.LISTING_ID column has a relationships test referencing the dim_listings_hosts.LISTING_ID column."
}
{
  "source_model": "fct_reviews",
  "source_column": "LISTING_ID",
  "expected_target_model": "dim_listings",
  "expected_target_column": "LISTING_ID",
  "declared_target_model": "dim_listings_hosts",
  "declared_target_column": "LISTING_ID",
  "confidence": "HIGH",
  "evidence": "The fct_reviews.LISTING_ID column has a relationships test referencing the mom_agg_reviews.date_sentiment_id column, but this appears to be

### 8.4 L1 — Missing NOT NULL Constraints

For the `L1_REMOVE_NOT_NULL` operator, the neural component reconstructs
attributes that are expected to satisfy a NOT NULL constraint from the
available schema evidence. The deterministic component independently
extracts the NOT NULL constraints explicitly declared in the mutated schema.

The two sets are then compared:

$$
Q_{\mathrm{missing}}
=
Q_{\mathrm{expected\_not\_null}}
-
Q_{\mathrm{declared\_not\_null}}
$$

A missing constraint is established when an expected NOT NULL proposition
has no corresponding explicit declaration:

$$
ExpectedNotNull(M,A)
\land
\neg DeclaredNotNull(M,A)
\rightarrow
MissingNotNull(M,A)
$$

The mutation metadata and ground truth are not exposed to the neural
component.

In [89]:
# V3 prompt — reconstruct EXPECTED_NOT_NULL expectations

V3_NOT_NULL_EXPECTATION_PROMPT = """You are analyzing a relational database project.

Your task is NOT to audit the schema for defects.

Given the supplied schema evidence, reconstruct which individual columns
are structurally expected to contain a value for every valid row.

Use only the supplied evidence.

Reconstruct an expected NOT NULL constraint only when there is specific
evidence that the column is structurally mandatory.

Strong evidence may include:
- the column is described as the identifier of each entity or row;
- the column participates in an explicitly declared uniqueness constraint
  and its semantics indicate that it identifies each represented entity;
- the column is explicitly described as required, mandatory, or always
  present.

A descriptive attribute is not expected to be NOT NULL merely because it
is useful or important for interpreting the row.

Do not infer NOT NULL merely because a column is:
- a name, description, date, coordinate, measure, or other descriptive
  attribute;
- an identifier or foreign key without additional evidence that it is
  mandatory;
- semantically important to the domain.

Do not identify missing constraints.
Do not compare expected constraints with declared NOT NULL tests.
Do not invent models or columns that are not present in the evidence.

Return only valid JSON using this structure:

{
  "expectations": [
    {
      "model": "<model name>",
      "column": "<column name>",
      "evidence": "<specific evidence supporting the expectation>",
      "confidence": "HIGH | MEDIUM | LOW"
    }
  ]
}

If no NOT NULL expectation can be established from the evidence, return:

{
  "expectations": []
}

Do not include markdown or text outside the JSON object.
"""

In [90]:
# Select one L1_REMOVE_NOT_NULL instance for sanity check

L1_NOT_NULL_BATCH_DIR = Path(
    "rdbscore_generated_v2/L1/L1_REMOVE_NOT_NULL"
)

l1_not_null_case_dirs = sorted([
    path
    for path in L1_NOT_NULL_BATCH_DIR.iterdir()
    if path.is_dir()
])

print(
    "L1_REMOVE_NOT_NULL instances:",
    len(l1_not_null_case_dirs)
)

l1_not_null_test_case = (
    l1_not_null_case_dirs[0]
)

print(
    "Selected case:",
    l1_not_null_test_case.name
)

with (
    l1_not_null_test_case /
    "ground_truth.json"
).open(
    "r",
    encoding="utf-8"
) as f:

    l1_not_null_test_gt = json.load(f)

print(
    json.dumps(
        l1_not_null_test_gt,
        indent=2,
        ensure_ascii=False
    )
)

L1_REMOVE_NOT_NULL instances: 861
Selected case: L1_REMOVE_NOT_NULL_0057fea9a08f
{
  "mutation_id": "L1_REMOVE_NOT_NULL_0057fea9a08f",
  "operator_id": "L1_REMOVE_NOT_NULL",
  "level": "L1",
  "defect_family": "Constraint Defects",
  "mutation_type": "REMOVE_NOT_NULL",
  "expected_defect": "Missing explicitly declared non-null constraint/test.",
  "evidence_requirement": "EXPLICIT_METADATA",
  "benchmark": "Spider2-DBT",
  "protocol_version": "RDBSCORE_V2",
  "project": "airport001",
  "artifact": "models/core/core.yml",
  "candidate": {
    "model": "fct_arrivals__malaysia_summary",
    "column": "airport_id"
  },
  "evidence": null
}


In [91]:
# V3 L1 NOT NULL — deterministic extraction of declared constraints

def extract_explicit_not_null_expectations(
    audit_input
):

    metadata = extract_schema_metadata(
        audit_input
    )

    expectations = []

    for model in metadata.get(
        "models", []
    ):

        model_name = model.get("name")

        for column in model.get(
            "columns", []
        ):

            column_name = column.get(
                "name"
            )

            tests = column.get(
                "tests", []
            ) or []

            has_not_null = any(
                (
                    test == "not_null"
                    or (
                        isinstance(test, dict)
                        and "not_null" in test
                    )
                )
                for test in tests
            )

            if has_not_null:

                expectations.append({
                    "model": model_name,
                    "column": column_name,
                    "derivation": "EXPLICIT"
                })

    return expectations

In [92]:
# V3 L1 NOT NULL — inspect selected mutated instance

l1_not_null_audit_path = (
    l1_not_null_test_case /
    "audit_input.txt"
)

l1_not_null_audit_input = (
    l1_not_null_audit_path.read_text(
        encoding="utf-8"
    )
)

l1_not_null_explicit = (
    extract_explicit_not_null_expectations(
        l1_not_null_audit_input
    )
)

target_model = (
    l1_not_null_test_gt[
        "candidate"
    ]["model"]
)

target_column = (
    l1_not_null_test_gt[
        "candidate"
    ]["column"]
)

target_explicit = any(
    (
        q["model"].lower()
        == target_model.lower()
        and
        q["column"].lower()
        == target_column.lower()
    )
    for q in l1_not_null_explicit
)

print(
    "Target:",
    f"{target_model}.{target_column}"
)

print(
    "Declared NOT NULL expectations:",
    len(l1_not_null_explicit)
)

print(
    "Target explicitly NOT NULL:",
    target_explicit
)

Target: fct_arrivals__malaysia_summary.airport_id
Declared NOT NULL expectations: 5
Target explicitly NOT NULL: False


In [93]:
# V3 L1 NOT NULL — neural reconstruction sanity check

l1_not_null_response = invoke_v3_sonnet(
    system_prompt=
        V3_NOT_NULL_EXPECTATION_PROMPT,
    evidence=
        l1_not_null_audit_input,
    max_tokens=2000
)

l1_not_null_result = (
    parse_v3_json_response(
        l1_not_null_response
    )
)

l1_not_null_expected = (
    l1_not_null_result.get(
        "expectations",
        []
    )
)

print(
    json.dumps(
        l1_not_null_expected,
        indent=2,
        ensure_ascii=False
    )
)

target_reconstructed = any(
    (
        q.get("model", "").lower()
        == target_model.lower()
        and
        q.get("column", "").lower()
        == target_column.lower()
    )
    for q in l1_not_null_expected
)

print(
    "\nTarget expectation reconstructed:",
    target_reconstructed
)

[
  {
    "model": "fct_arrivals__malaysia_summary",
    "column": "airport_id",
    "evidence": "Unique OpenFlights identifier for each airport.",
    "confidence": "HIGH"
  },
  {
    "model": "fct_arrivals__malaysia_summary",
    "column": "name",
    "evidence": "Name of the airport. May or may not contain the city name.",
    "confidence": "HIGH"
  },
  {
    "model": "fct_arrivals__malaysia_summary",
    "column": "latitude",
    "evidence": "Latitude of the airport in decimal degrees. Negative is South, positive is North.",
    "confidence": "HIGH"
  },
  {
    "model": "fct_arrivals__malaysia_summary",
    "column": "longitude",
    "evidence": "Longitude of the airport in decimal degrees. Negative is West, positive is East.",
    "confidence": "HIGH"
  },
  {
    "model": "fct_arrivals__malaysia_summary",
    "column": "flight_count",
    "evidence": "The total number of flight arrivals to each airport on the most recent available date, excluding code-share flights.",
    "con

In [102]:
# V3 batch audit — L1_REMOVE_NOT_NULL

def not_null_expectation_key(q):

    return (
        str(q.get("model", "")).strip().lower(),
        str(q.get("column", "")).strip().lower()
    )


def audit_v3_remove_not_null(audit_input):

    # 1. Neural reconstruction of expected NOT NULL constraints
    response = invoke_v3_sonnet(
        system_prompt=
            V3_NOT_NULL_EXPECTATION_PROMPT,
        evidence=audit_input,
        max_tokens=8192,
    )

    result = parse_v3_json_response(
        response
    )

    q_expected = result.get(
        "expectations",
        []
    )

    # 2. Deterministic extraction of declared NOT NULL constraints
    q_explicit = (
        extract_explicit_not_null_expectations(
            audit_input
        )
    )

    # 3. Deterministic set comparison
    explicit_keys = {
        not_null_expectation_key(q)
        for q in q_explicit
    }

    q_missing = [
        q
        for q in q_expected
        if (
            not_null_expectation_key(q)
            not in explicit_keys
        )
    ]

    # 4. Produce grounded findings
    findings = []

    for q in q_missing:

        findings.append({
            "finding":
                "MISSING_NOT_NULL",

            "model":
                q.get("model"),

            "column":
                q.get("column"),

            "evidence":
                q.get("evidence"),

            "confidence":
                q.get("confidence")
        })

    return {
        "findings":
            findings,

        "reconstructed_expectations":
            q_expected,

        "explicit_expectations":
            q_explicit,

        "raw_response":
            response
    }

In [103]:
# Sanity check — complete L1_REMOVE_NOT_NULL audit

l1_not_null_audit_result = (
    audit_v3_remove_not_null(
        l1_not_null_audit_input
    )
)

print(
    "TARGET:",
    f"{target_model}.{target_column}"
)

print(
    "\nFINDINGS:"
)

print(
    json.dumps(
        l1_not_null_audit_result[
            "findings"
        ],
        indent=2,
        ensure_ascii=False
    )
)

target_detected = any(
    (
        finding.get(
            "finding"
        )
        == "MISSING_NOT_NULL"

        and finding.get(
            "model", ""
        ).lower()
        == target_model.lower()

        and finding.get(
            "column", ""
        ).lower()
        == target_column.lower()
    )
    for finding
    in l1_not_null_audit_result[
        "findings"
    ]
)

print(
    "\nTarget detected:",
    target_detected
)

print(
    "Total findings:",
    len(
        l1_not_null_audit_result[
            "findings"
        ]
    )
)

TARGET: fct_arrivals__malaysia_summary.airport_id

FINDINGS:
[
  {
    "finding": "MISSING_NOT_NULL",
    "model": "fct_arrivals__malaysia_summary",
    "column": "airport_id",
    "evidence": "Unique OpenFlights identifier for each airport.",
    "confidence": "HIGH"
  }
]

Target detected: True
Total findings: 1


In [104]:
# L1_REMOVE_NOT_NULL batch setup

L1_NOT_NULL_V3_RESULTS_PATH = Path(
    "v3_l1_remove_not_null_results.jsonl"
)

l1_not_null_case_dirs = sorted([
    path
    for path in L1_NOT_NULL_BATCH_DIR.iterdir()
    if (
        path.is_dir()
        and (
            path / "audit_input.txt"
        ).exists()
    )
])

print(
    "L1_REMOVE_NOT_NULL instances:",
    len(l1_not_null_case_dirs)
)

print(
    "Results file:",
    L1_NOT_NULL_V3_RESULTS_PATH
)

L1_REMOVE_NOT_NULL instances: 861
Results file: v3_l1_remove_not_null_results.jsonl


In [105]:
# Run V3 batch — L1_REMOVE_NOT_NULL
# Incremental JSONL persistence with resume support

completed_l1_not_null_cases = set()

if L1_NOT_NULL_V3_RESULTS_PATH.exists():

    with L1_NOT_NULL_V3_RESULTS_PATH.open(
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:

            if not line.strip():
                continue

            try:
                record = json.loads(line)

                if record.get("status") == "OK":
                    completed_l1_not_null_cases.add(
                        record["case_id"]
                    )

            except json.JSONDecodeError:
                pass


remaining_l1_not_null_cases = [
    case_dir
    for case_dir in l1_not_null_case_dirs
    if (
        case_dir.name
        not in completed_l1_not_null_cases
    )
]


print(
    "Already completed:",
    len(completed_l1_not_null_cases)
)

print(
    "Remaining:",
    len(remaining_l1_not_null_cases)
)


with L1_NOT_NULL_V3_RESULTS_PATH.open(
    "a",
    encoding="utf-8"
) as output_file:

    for index, case_dir in enumerate(
        remaining_l1_not_null_cases,
        start=1
    ):

        case_id = case_dir.name

        try:

            audit_input = (
                case_dir /
                "audit_input.txt"
            ).read_text(
                encoding="utf-8"
            )

            result = (
                audit_v3_remove_not_null(
                    audit_input
                )
            )

            record = {
                "case_id":
                    case_id,

                "status":
                    "OK",

                "findings":
                    result["findings"],

                "reconstructed_expectations":
                    result[
                        "reconstructed_expectations"
                    ],

                "explicit_expectations":
                    result[
                        "explicit_expectations"
                    ]
            }

            output_file.write(
                json.dumps(
                    record,
                    ensure_ascii=False
                )
                + "\n"
            )

            output_file.flush()

            print(
                f"[{index}/"
                f"{len(remaining_l1_not_null_cases)}] "
                f"OK {case_id} "
                f"findings="
                f"{len(result['findings'])}"
            )

        except Exception as e:

            record = {
                "case_id":
                    case_id,

                "status":
                    "ERROR",

                "error":
                    repr(e)
            }

            output_file.write(
                json.dumps(
                    record,
                    ensure_ascii=False
                )
                + "\n"
            )

            output_file.flush()

            print(
                f"[{index}/"
                f"{len(remaining_l1_not_null_cases)}] "
                f"ERROR {case_id}: {e}"
            )

Already completed: 857
Remaining: 4
[1/4] OK L1_REMOVE_NOT_NULL_08f00af77369 findings=0
[2/4] ERROR L1_REMOVE_NOT_NULL_7ca296432482: Invalid JSON model response: 'Based on the provided schema metadata and SQL model definitions, here are the columns that appear to be structurally expected to contain a value for every valid row, along with the supporting evidence and confidence level:\n\n{\n  "expectations": [\n    {\n      "model": "stg_workday__personal_information_history",\n      "column": "history_unique_key",\n      "evidence": "Surrogate key hashed on `worker_id`, `source_relation` and `_fivetran_start`.",\n      "confidence": "HIGH"\n    },\n    {\n      "mode'
[3/4] ERROR L1_REMOVE_NOT_NULL_8739cf172ff1: Invalid JSON model response: 'Based on the provided schema metadata and SQL model definitions, here are the columns that appear to be structurally expected to contain a value for every valid row, along with the supporting evidence and confidence level:\n\n{\n  "expectations": [\

In [106]:
# Consolidate L1_REMOVE_NOT_NULL batch results

latest_l1_not_null_ok_by_case = {}
l1_not_null_errors = []

with L1_NOT_NULL_V3_RESULTS_PATH.open(
    "r",
    encoding="utf-8"
) as f:

    for line in f:

        if not line.strip():
            continue

        record = json.loads(line)

        if record.get("status") == "OK":
            latest_l1_not_null_ok_by_case[
                record["case_id"]
            ] = record

        elif record.get("status") == "ERROR":
            l1_not_null_errors.append(record)


expected_case_ids = {
    case_dir.name
    for case_dir in l1_not_null_case_dirs
}

successful_case_ids = set(
    latest_l1_not_null_ok_by_case
)

missing_case_ids = (
    expected_case_ids - successful_case_ids
)

print("=== V3 L1 NOT NULL CONSOLIDATION ===")
print("Expected cases:", len(expected_case_ids))
print("Successful unique cases:", len(successful_case_ids))
print("Error records in JSONL:", len(l1_not_null_errors))
print("Cases without successful result:", len(missing_case_ids))

if missing_case_ids:
    print("\nCases requiring retry:")
    for case_id in sorted(missing_case_ids):
        print(case_id)

=== V3 L1 NOT NULL CONSOLIDATION ===
Expected cases: 861
Successful unique cases: 858
Error records in JSONL: 933
Cases without successful result: 3

Cases requiring retry:
L1_REMOVE_NOT_NULL_7ca296432482
L1_REMOVE_NOT_NULL_8739cf172ff1
L1_REMOVE_NOT_NULL_a9e53e4f16d0


In [107]:
# Diagnose L1_REMOVE_NOT_NULL batch errors

from collections import Counter

pending_errors = [
    record
    for record in l1_not_null_errors
    if record["case_id"] in missing_case_ids
]

error_counts = Counter(
    record["error"]
    for record in pending_errors
)

print("=== ERROR DIAGNOSIS ===")
print("Pending cases:", len(missing_case_ids))
print("Error attempts for pending cases:", len(pending_errors))
print("Distinct error messages:", len(error_counts))

print("\nMost frequent errors:")

for error, count in error_counts.most_common(10):
    print(f"\nOccurrences: {count}")
    print(error[:700])

=== ERROR DIAGNOSIS ===
Pending cases: 3
Error attempts for pending cases: 18
Distinct error messages: 4

Most frequent errors:

Occurrences: 10
ValueError('Invalid JSON model response: \'Based on the provided schema metadata and SQL model definitions, here are the columns that appear to be structurally expected to contain a value for every valid row, along with the supporting evidence and confidence level:\\n\\n{\\n  "expectations": [\\n    {\\n      "model": "stg_workday__personal_information_history",\\n      "column": "history_unique_key",\\n      "evidence": "Surrogate key hashed on `worker_id`, `source_relation` and `_fivetran_start`.",\\n      "confidence": "HIGH"\\n    },\\n    {\\n      "mode\'')

Occurrences: 4
ValueError('Invalid JSON model response: \'Based on the provided schema metadata and SQL model definitions, here are the columns that appear to be structurally expected to contain a value for every valid row, along with the supporting evidence and confidence level:\\n\

In [108]:
# Diagnose distribution of failed L1 NOT NULL cases

from collections import Counter

failed_case_attempts = Counter(
    record["case_id"]
    for record in pending_errors
)

print("=== FAILED CASE DISTRIBUTION ===")

print("Pending unique cases:", len(failed_case_attempts))

print(
    "Attempts per case:",
    Counter(failed_case_attempts.values())
)

print("\nSample pending cases:")

for case_id, attempts in failed_case_attempts.most_common(10):
    print(case_id, "attempts:", attempts)

=== FAILED CASE DISTRIBUTION ===
Pending unique cases: 3
Attempts per case: Counter({6: 3})

Sample pending cases:
L1_REMOVE_NOT_NULL_7ca296432482 attempts: 6
L1_REMOVE_NOT_NULL_8739cf172ff1 attempts: 6
L1_REMOVE_NOT_NULL_a9e53e4f16d0 attempts: 6


In [129]:
# Refresh L1_REMOVE_NOT_NULL completion status

latest_l1_not_null_ok_by_case = {}
latest_error_by_case = {}

with L1_NOT_NULL_V3_RESULTS_PATH.open(
    "r",
    encoding="utf-8"
) as f:

    for line in f:
        if not line.strip():
            continue

        record = json.loads(line)
        case_id = record["case_id"]

        if record.get("status") == "OK":
            latest_l1_not_null_ok_by_case[case_id] = record

        elif record.get("status") == "ERROR":
            latest_error_by_case[case_id] = record

expected_case_ids = {
    case_dir.name
    for case_dir in l1_not_null_case_dirs
}

missing_case_ids = (
    expected_case_ids
    - set(latest_l1_not_null_ok_by_case)
)

print("=== CURRENT L1 NOT NULL STATUS ===")
print("Expected:", len(expected_case_ids))
print("Successful:", len(latest_l1_not_null_ok_by_case))
print("Pending:", len(missing_case_ids))

print("\nPending cases:")
for case_id in sorted(missing_case_ids):
    error = latest_error_by_case.get(case_id, {})
    print(case_id, error.get("error", "No recorded error")[:250])

=== CURRENT L1 NOT NULL STATUS ===
Expected: 861
Successful: 859
Pending: 2

Pending cases:
L1_REMOVE_NOT_NULL_7ca296432482 ValueError('Invalid JSON model response: \'Based on the provided schema metadata and SQL model definitions, here are the columns that appear to be structurally expected to contain a value for every valid row, along with the supporting evidence and co
L1_REMOVE_NOT_NULL_a9e53e4f16d0 ValueError('Invalid JSON model response: \'Based on the provided schema metadata and SQL model definitions, here are the columns that appear to be structurally expected to contain a value for every valid row, along with the supporting evidence and co


In [130]:
# Inspect remaining L1 NOT NULL failures and input sizes

print("=== REMAINING L1 NOT NULL CASES ===")

for case_id in sorted(missing_case_ids):

    case_dir = L1_NOT_NULL_BATCH_DIR / case_id

    audit_input = (
        case_dir / "audit_input.txt"
    ).read_text(encoding="utf-8")

    metadata = extract_schema_metadata(audit_input)

    models = metadata.get("models", [])

    columns = sum(
        len(model.get("columns", []))
        for model in models
    )

    print("\nCase:", case_id)
    print("Input characters:", len(audit_input))
    print("Models:", len(models))
    print("Columns:", columns)

    error = latest_error_by_case.get(case_id, {})

    print(
        "Last error:",
        error.get("error", "")[:350]
    )

=== REMAINING L1 NOT NULL CASES ===

Case: L1_REMOVE_NOT_NULL_7ca296432482
Input characters: 31770
Models: 4
Columns: 209
Last error: ValueError('Invalid JSON model response: \'Based on the provided schema metadata and SQL model definitions, here are the columns that appear to be structurally expected to contain a value for every valid row, along with the supporting evidence and confidence level:\\n\\n{\\n  "expectations": [\\n    {\\n      "model": "stg_workday__personal_informa

Case: L1_REMOVE_NOT_NULL_a9e53e4f16d0
Input characters: 31770
Models: 4
Columns: 209
Last error: ValueError('Invalid JSON model response: \'Based on the provided schema metadata and SQL model definitions, here are the columns that appear to be structurally expected to contain a value for every valid row, along with the supporting evidence and confidence level:\\n\\n{\\n  "expectations": [\\n    {\\n      "model": "stg_workday__personal_informa


In [131]:
# V3 L1 NOT NULL — partition evidence by model and column groups

def audit_v3_remove_not_null_partitioned(
    audit_input,
    columns_per_group=20
):

    metadata = extract_schema_metadata(audit_input)

    all_expectations = []
    responses = []

    for model in metadata.get("models", []):

        columns = model.get("columns", []) or []

        for start in range(
            0,
            len(columns),
            columns_per_group
        ):

            group = columns[
                start:start + columns_per_group
            ]

            model_subset = {
                **model,
                "columns": group
            }

            evidence_metadata = {
                **metadata,
                "models": [model_subset]
            }

            evidence = (
                "SCHEMA METADATA\n"
                "========================================\n"
                + yaml.safe_dump(
                    evidence_metadata,
                    sort_keys=False,
                    allow_unicode=True
                )
            )

            response = invoke_v3_sonnet(
                system_prompt=V3_NOT_NULL_EXPECTATION_PROMPT,
                evidence=evidence,
                max_tokens=4096
            )

            result = parse_v3_json_response(response)

            all_expectations.extend(
                result.get("expectations", [])
            )

            responses.append({
                "model": model.get("name"),
                "column_group_start": start,
                "raw_response": response
            })

    # Deterministic extraction from complete original schema
    q_explicit = extract_explicit_not_null_expectations(
        audit_input
    )

    explicit_keys = {
        not_null_expectation_key(q)
        for q in q_explicit
    }

    expected_by_key = {
        not_null_expectation_key(q): q
        for q in all_expectations
    }

    q_expected = list(expected_by_key.values())

    q_missing = [
        q
        for q in q_expected
        if not_null_expectation_key(q) not in explicit_keys
    ]

    findings = [
        {
            "finding": "MISSING_NOT_NULL",
            "model": q["model"],
            "column": q["column"],
            "evidence": q.get("evidence"),
            "confidence": q.get("confidence")
        }
        for q in q_missing
    ]

    return {
        "findings": findings,
        "reconstructed_expectations": q_expected,
        "explicit_expectations": q_explicit,
        "raw_responses": responses
    }

In [132]:
# V3 JSON parsing — handle unescaped dbt doc() references

import re

_original_parse_v3_json_response = parse_v3_json_response

def parse_v3_json_response(response):

    try:
        return _original_parse_v3_json_response(response)

    except ValueError:

        # Escape quotes inside dbt doc() references
        repaired = re.sub(
            r'\{\{\s*doc\("([^"]+)"\)\s*\}\}',
            lambda match: (
                '{{ doc(\\"'
                + match.group(1)
                + '\\") }}'
            ),
            response
        )

        return _original_parse_v3_json_response(repaired)

In [133]:
# Verify JSON repair using the captured response

parsed = parse_v3_json_response(response)

print("JSON VALID")
print(
    json.dumps(
        parsed,
        indent=2,
        ensure_ascii=False
    )
)

JSON VALID
{
  "expectations": [
    {
      "model": "stg_workday__personal_information_history",
      "column": "worker_id",
      "evidence": "The column is described as '{{ doc(\"worker_id\") }}' and has a not_null test, indicating it is a required identifier for each worker record.",
      "confidence": "HIGH"
    },
    {
      "model": "stg_workday__personal_information_history",
      "column": "history_unique_key",
      "evidence": "The column is described as a 'Surrogate key hashed on `worker_id`, `source_relation` and `_fivetran_start`' and has a unique test, indicating it uniquely identifies each history record.",
      "confidence": "HIGH"
    }
  ]
}


In [134]:
# Complete remaining L1_REMOVE_NOT_NULL cases using partitioned evidence

for case_id in sorted(missing_case_ids):

    case_dir = L1_NOT_NULL_BATCH_DIR / case_id

    print("Processing:", case_id)

    try:
        audit_input = (
            case_dir / "audit_input.txt"
        ).read_text(encoding="utf-8")

        result = audit_v3_remove_not_null_partitioned(
            audit_input
        )

        record = {
            "case_id": case_id,
            "status": "OK",
            "processing_mode": "PARTITIONED_BY_MODEL",
            "findings": result["findings"],
            "reconstructed_expectations":
                result["reconstructed_expectations"],
            "explicit_expectations":
                result["explicit_expectations"]
        }

        with L1_NOT_NULL_V3_RESULTS_PATH.open(
            "a",
            encoding="utf-8"
        ) as f:
            f.write(
                json.dumps(record, ensure_ascii=False)
                + "\n"
            )

        print(
            "OK — findings:",
            len(result["findings"])
        )

    except Exception as e:
        print("ERROR:", repr(e))

Processing: L1_REMOVE_NOT_NULL_7ca296432482
OK — findings: 4
Processing: L1_REMOVE_NOT_NULL_a9e53e4f16d0
OK — findings: 4


In [136]:
# V3 evaluation — L1_REMOVE_NOT_NULL

latest_l1_not_null_ok_by_case = {}

with L1_NOT_NULL_V3_RESULTS_PATH.open(
    "r",
    encoding="utf-8"
) as f:

    for line in f:
        if not line.strip():
            continue

        record = json.loads(line)

        if record.get("status") == "OK":
            latest_l1_not_null_ok_by_case[
                record["case_id"]
            ] = record


evaluation = []

for case_dir in l1_not_null_case_dirs:

    case_id = case_dir.name

    if case_id not in latest_l1_not_null_ok_by_case:
        continue

    with (case_dir / "ground_truth.json").open(
        "r",
        encoding="utf-8"
    ) as f:
        gt = json.load(f)

    target_model = gt["candidate"]["model"]
    target_column = gt["candidate"]["column"]

    record = latest_l1_not_null_ok_by_case[case_id]
    findings = record.get("findings", [])

    target_detected = any(
        finding.get("finding") == "MISSING_NOT_NULL"
        and not_null_expectation_key(finding)
        == (
            target_model.strip().lower(),
            target_column.strip().lower()
        )
        for finding in findings
    )

    evaluation.append({
        "case_id": case_id,
        "target_detected": target_detected,
        "exact_case_match": (
            target_detected and len(findings) == 1
        ),
        "finding_count": len(findings),
        "processing_mode": record.get(
            "processing_mode", "FULL_EVIDENCE"
        )
    })


n = len(evaluation)
detected = sum(r["target_detected"] for r in evaluation)
exact = sum(r["exact_case_match"] for r in evaluation)

print("=== V3 L1 NOT NULL EVALUATION ===")
print("N:", n)

if n:
    print(
        f"Target detected: {detected} "
        f"({100 * detected / n:.2f}%)"
    )
    print(
        f"Exact case match: {exact} "
        f"({100 * exact / n:.2f}%)"
    )
    print(
        f"Target not detected: {n - detected} "
        f"({100 * (n - detected) / n:.2f}%)"
    )

print(
    "\nProcessing modes:",
    dict(Counter(r["processing_mode"] for r in evaluation))
)

=== V3 L1 NOT NULL EVALUATION ===
N: 861
Target detected: 128 (14.87%)
Exact case match: 110 (12.78%)
Target not detected: 733 (85.13%)

Processing modes: {'FULL_EVIDENCE': 858, 'PARTITIONED_BY_MODEL': 3}
